In [1]:
#!/usr/bin/env python3
"""
y4_stage0_geometry_manifest.py
================================

Self-contained Stage-0 compiler for the O(y^4) SU(3) one-flux flat-band program.

PURPOSE
-------
This script freezes the finite CONNECTED geometry problem before any expensive
SU(3) Haar/representation contraction is attempted.

It does all of the following in exact integer/rational arithmetic:

  1. Regresses the certified O(y^3) arithmetic anchors
       b3      =  1975/124848
       leak3   = -12331/249696
       d3      = 7/32 + 12 leak3 - 4 b3
               = -109151/249696

  2. Enumerates every rooted, site-connected multiset of FOUR plaquette
     perturbation insertions on the cubic lattice, INCLUDING repetitions.

  3. Quotients those supports by the 8-element proper cubic stabilizer of the
     rooted input xy plaquette.

  4. Attaches every possible one-plaquette output whose boundary links are
     contained in the geometric support.

  5. Applies an exact NECESSARY SU(3) link-triality/bare-link filter:
       total fundamental-minus-antifundamental incidence == 0 mod 3
     on every link, after enumerating all 2^6 orientation assignments
     (ket, four insertions, bra).

  6. Expands surviving support classes into ordered fourth-order words and
     quotients the ordered transitions by rooted proper cubic symmetry.

  7. Writes frozen gzip JSON manifests with SHA-256 hashes.

IMPORTANT SCOPE
---------------
This is NOT yet the fourth-order Haar contraction and does NOT compute w_g.

It is the correct finite candidate compiler for the next stage. It deliberately
keeps:
  * repeated plaquette insertions,
  * site-only corner contacts,
  * all ordered words surviving the necessary triality filter.

The later des-Cloizeaux/linked-cluster stage must still add:
  * exact intermediate SU(3) representation channels,
  * electric-energy denominators,
  * folded/subtraction terms required by the chosen effective Hamiltonian,
  * exact Haar contractions and rational weights.

RUNTIME / HARDWARE
------------------
Use a standard Colab CPU runtime. An A100 is not used by this stage.
Typical runtime is tens of seconds to a few minutes.

COLAB
-----
Upload this file, then run exactly:

    %run /content/y4_stage0_geometry_manifest.py

Outputs are written to:

    /content/Y4_STAGE0/

No edits are required.
"""

from __future__ import annotations

import argparse
import gzip
import hashlib
import itertools
import json
import math
import os
import platform
import sys
import time
from collections import Counter, defaultdict
from fractions import Fraction
from functools import lru_cache
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Set, Tuple

VERSION = "2026-06-13-stage0-v1"

# ---------------------------------------------------------------------------
# Paths / runtime
# ---------------------------------------------------------------------------

def default_output_dir() -> Path:
    if Path("/content").exists():
        return Path("/content/Y4_STAGE0")
    return Path.cwd() / "Y4_STAGE0"


# ---------------------------------------------------------------------------
# Exact cubic-lattice geometry
#
# Plaquette representation:
#   (x, y, z, a, b), with 0 <= a < b <= 2
# where (x,y,z) is the lower anchor and a,b are coordinate axes.
#
# Link representation:
#   (x, y, z, a)
# where (x,y,z) is the lower anchor and a is the positive axis.
# ---------------------------------------------------------------------------

Vec3 = Tuple[int, int, int]
Plaquette = Tuple[int, int, int, int, int]
Link = Tuple[int, int, int, int]

E: Tuple[Vec3, Vec3, Vec3] = (
    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
)

PLANES = ((0, 1), (0, 2), (1, 2))
PLANE_NAME = {(0, 1): "xy", (0, 2): "xz", (1, 2): "yz"}

ROOT: Plaquette = (0, 0, 0, 0, 1)


def vadd(u: Vec3, v: Vec3) -> Vec3:
    return (u[0] + v[0], u[1] + v[1], u[2] + v[2])


def vsub(u: Vec3, v: Vec3) -> Vec3:
    return (u[0] - v[0], u[1] - v[1], u[2] - v[2])


def smul(s: int, v: Vec3) -> Vec3:
    return (s * v[0], s * v[1], s * v[2])


@lru_cache(maxsize=None)
def vertices(p: Plaquette) -> Tuple[Vec3, Vec3, Vec3, Vec3]:
    x = p[:3]
    a, b = p[3], p[4]
    return (
        x,
        vadd(x, E[a]),
        vadd(x, E[b]),
        vadd(vadd(x, E[a]), E[b]),
    )


@lru_cache(maxsize=None)
def boundary(p: Plaquette) -> Tuple[Tuple[Link, int], ...]:
    """Positively oriented plaquette boundary: +a,+b,-a,-b."""
    x = p[:3]
    a, b = p[3], p[4]

    xa = vadd(x, E[a])
    xb = vadd(x, E[b])

    return (
        ((x[0], x[1], x[2], a), +1),
        ((xa[0], xa[1], xa[2], b), +1),
        ((xb[0], xb[1], xb[2], a), -1),
        ((x[0], x[1], x[2], b), -1),
    )


@lru_cache(maxsize=None)
def link_set(p: Plaquette) -> frozenset[Link]:
    return frozenset(link for link, _ in boundary(p))


@lru_cache(maxsize=None)
def vertex_set(p: Plaquette) -> frozenset[Vec3]:
    return frozenset(vertices(p))


def plaquettes_containing_vertices(vs: Iterable[Vec3]) -> Set[Plaquette]:
    """All unit plaquettes containing at least one supplied lattice vertex."""
    out: Set[Plaquette] = set()
    for v in vs:
        for a, b in PLANES:
            for ia in (0, 1):
                for ib in (0, 1):
                    anchor = vsub(vsub(v, smul(ia, E[a])), smul(ib, E[b]))
                    out.add((*anchor, a, b))
    return out


# ---------------------------------------------------------------------------
# Proper cubic rotations and rooted stabilizer
# ---------------------------------------------------------------------------

Rotation = Tuple[Tuple[int, int, int], Tuple[int, int, int]]


def permutation_parity(perm: Tuple[int, int, int]) -> int:
    inv = sum(
        perm[i] > perm[j]
        for i in range(3)
        for j in range(i + 1, 3)
    )
    return -1 if inv % 2 else +1


def proper_cubic_rotations() -> List[Rotation]:
    out: List[Rotation] = []
    for perm in itertools.permutations(range(3)):
        parity = permutation_parity(perm)
        for signs in itertools.product((-1, +1), repeat=3):
            if parity * signs[0] * signs[1] * signs[2] == +1:
                out.append((perm, signs))
    assert len(out) == 24
    return out


ROTATIONS: List[Rotation] = proper_cubic_rotations()


def transform_vec(v: Vec3, rot: Rotation) -> Vec3:
    perm, signs = rot
    out = [0, 0, 0]
    for j in range(3):
        out[perm[j]] += signs[j] * v[j]
    return (out[0], out[1], out[2])


@lru_cache(maxsize=None)
def transform_plaquette_cached(p: Plaquette, rotation_index: int) -> Plaquette:
    perm, signs = ROTATIONS[rotation_index]
    anchor = p[:3]
    a, b = p[3], p[4]

    aa, bb = perm[a], perm[b]
    new_anchor = list(transform_vec(anchor, (perm, signs)))

    if signs[a] < 0:
        new_anchor[aa] -= 1
    if signs[b] < 0:
        new_anchor[bb] -= 1

    aa, bb = sorted((aa, bb))
    return (new_anchor[0], new_anchor[1], new_anchor[2], aa, bb)


ROOT_STABILIZER: List[int] = [
    i
    for i in range(len(ROTATIONS))
    if transform_plaquette_cached(ROOT, i)[3:] == (0, 1)
]

assert len(ROOT_STABILIZER) == 8

ROOT_SHIFT: Dict[int, Vec3] = {
    i: transform_plaquette_cached(ROOT, i)[:3]
    for i in ROOT_STABILIZER
}


@lru_cache(maxsize=None)
def rooted_transform(p: Plaquette, rotation_index: int) -> Plaquette:
    q = transform_plaquette_cached(p, rotation_index)
    t = ROOT_SHIFT[rotation_index]
    return (q[0] - t[0], q[1] - t[1], q[2] - t[2], q[3], q[4])


def canonical_multiset(ms: Sequence[Plaquette]) -> Tuple[Plaquette, ...]:
    return min(
        tuple(sorted(rooted_transform(p, i) for p in ms))
        for i in ROOT_STABILIZER
    )


def canonical_support_output(
    ms: Sequence[Plaquette],
    output: Plaquette,
) -> Tuple[Tuple[Plaquette, ...], Plaquette]:
    return min(
        (
            tuple(sorted(rooted_transform(p, i) for p in ms)),
            rooted_transform(output, i),
        )
        for i in ROOT_STABILIZER
    )


def canonical_ordered_transition(
    word: Sequence[Plaquette],
    output: Plaquette,
) -> Tuple[Tuple[Plaquette, ...], Plaquette]:
    return min(
        (
            tuple(rooted_transform(p, i) for p in word),
            rooted_transform(output, i),
        )
        for i in ROOT_STABILIZER
    )


# ---------------------------------------------------------------------------
# Support enumeration
# ---------------------------------------------------------------------------

def enumerate_connected_multisets(
    order: int = 4,
) -> Tuple[Set[Tuple[Plaquette, ...]], List[int]]:
    """
    Enumerate rooted site-connected multisets of `order` plaquette insertions.

    Repetition is allowed. Connectivity is enforced constructively: every new
    plaquette contains at least one vertex already in the rooted support.
    """
    states: Set[Tuple[Plaquette, ...]] = {()}
    counts = [1]

    for depth in range(1, order + 1):
        started = time.time()
        new_states: Set[Tuple[Plaquette, ...]] = set()

        for ms in states:
            support_vertices: Set[Vec3] = set(vertices(ROOT))
            for p in ms:
                support_vertices.update(vertices(p))

            for p in plaquettes_containing_vertices(support_vertices):
                new_states.add(canonical_multiset(ms + (p,)))

        states = new_states
        counts.append(len(states))

        print(
            f"[enumerate] depth={depth} classes={len(states):,} "
            f"elapsed={time.time() - started:.2f}s",
            flush=True,
        )

    return states, counts


def candidate_outputs(ms: Sequence[Plaquette]) -> Tuple[Plaquette, ...]:
    """
    Candidate one-plaquette outputs.

    Any output link absent from ROOT + insertions would be a lone external
    fundamental/antifundamental factor and fail the triality test. Therefore
    it is necessary that all four output boundary links lie in the union.
    """
    links: Set[Link] = set()
    support_vertices: Set[Vec3] = set(vertices(ROOT))

    for p in (ROOT,) + tuple(ms):
        support_vertices.update(vertices(p))
        links.update(link for link, _ in boundary(p))

    out: Set[Plaquette] = set()
    for anchor in support_vertices:
        for a, b in PLANES:
            p = (*anchor, a, b)
            if all(link in links for link, _ in boundary(p)):
                out.add(p)

    return tuple(sorted(out))


# ---------------------------------------------------------------------------
# Exact SU(3) triality / bare-link filter
#
# Variables:
#   s0      ket orientation
#   s1..s4 perturbation character orientations
#   s5      bra orientation
#
# Each s is ±1. The bra contribution is conjugated, hence carries -s5.
# A necessary SU(3) Haar condition is zero net link triality mod 3.
#
# We precompute, for each row in F_3^6, a 64-bit mask of sign patterns that
# satisfy row dot sign == 0 mod 3.
# ---------------------------------------------------------------------------

MOD3_SIGN_PATTERNS: List[Tuple[int, ...]] = list(
    itertools.product((1, 2), repeat=6)
)
SIGNED_PATTERNS: List[Tuple[int, ...]] = list(
    itertools.product((-1, +1), repeat=6)
)


def build_row_masks() -> List[int]:
    masks = [0] * (3 ** 6)

    for code in range(3 ** 6):
        x = code
        row: List[int] = []
        for _ in range(6):
            row.append(x % 3)
            x //= 3

        mask = 0
        for index, signs in enumerate(MOD3_SIGN_PATTERNS):
            if sum(row[j] * signs[j] for j in range(6)) % 3 == 0:
                mask |= 1 << index

        masks[code] = mask

    return masks


ROW_MASK = build_row_masks()
ALL_SIGN_MASK = (1 << 64) - 1


def admissible_sign_mask(
    ms: Sequence[Plaquette],
    output: Plaquette,
) -> int:
    assert len(ms) == 4

    rows: Dict[Link, List[int]] = {}

    factors = (ROOT,) + tuple(ms) + (output,)

    for column, p in enumerate(factors):
        external_factor = -1 if column == 5 else +1

        for link, incidence in boundary(p):
            row = rows.setdefault(link, [0] * 6)
            row[column] += external_factor * incidence

    mask = ALL_SIGN_MASK

    for row in rows.values():
        code = 0
        multiplier = 1

        for value in row:
            code += (value % 3) * multiplier
            multiplier *= 3

        mask &= ROW_MASK[code]

        if mask == 0:
            return 0

    return mask


def signed_patterns_from_mask(mask: int) -> List[List[int]]:
    return [
        list(SIGNED_PATTERNS[i])
        for i in range(64)
        if (mask >> i) & 1
    ]


# ---------------------------------------------------------------------------
# Classification
# ---------------------------------------------------------------------------

def ordered_word_multiplicity(ms: Sequence[Plaquette]) -> int:
    counts = Counter(ms)
    value = math.factorial(len(ms))

    for multiplicity in counts.values():
        value //= math.factorial(multiplicity)

    return value


def support_extent(plaqs: Sequence[Plaquette]) -> List[int]:
    vs: Set[Vec3] = set()

    for p in plaqs:
        vs.update(vertices(p))

    return [
        max(v[axis] for v in vs) - min(v[axis] for v in vs)
        for axis in range(3)
    ]


def contact_flags(plaqs: Sequence[Plaquette]) -> Tuple[bool, bool]:
    unique = tuple(sorted(set(plaqs)))
    has_link = False
    has_site_only = False

    for i in range(len(unique)):
        for j in range(i + 1, len(unique)):
            if link_set(unique[i]) & link_set(unique[j]):
                has_link = True
            elif vertex_set(unique[i]) & vertex_set(unique[j]):
                has_site_only = True

    return has_link, has_site_only


def json_plaquette(p: Plaquette) -> List[int]:
    return [int(x) for x in p]


def class_id(
    ms: Sequence[Plaquette],
    output: Plaquette,
) -> str:
    payload = json.dumps(
        {
            "operators": [json_plaquette(p) for p in ms],
            "output": json_plaquette(output),
        },
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()[:20]


# ---------------------------------------------------------------------------
# JSON helpers
# ---------------------------------------------------------------------------

def write_json(path: Path, obj: object) -> str:
    data = json.dumps(
        obj,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ).encode("utf-8")

    path.write_bytes(data)
    return hashlib.sha256(data).hexdigest()


def write_json_gz(path: Path, obj: object) -> str:
    raw = json.dumps(
        obj,
        separators=(",", ":"),
        sort_keys=True,
        allow_nan=False,
    ).encode("utf-8")

    with gzip.GzipFile(
        filename=str(path),
        mode="wb",
        compresslevel=9,
        mtime=0,
    ) as handle:
        handle.write(raw)

    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_json_gz(path: Path) -> object:
    with gzip.open(path, "rb") as handle:
        return json.loads(handle.read().decode("utf-8"))


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main(output_dir: Path) -> None:
    started = time.time()
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 96)
    print("SU(3) O(y^4) STAGE-0 CONNECTED GEOMETRY / TRIALITY MANIFEST")
    print("=" * 96)
    print(f"version       : {VERSION}")
    print(f"python        : {sys.version.split()[0]}")
    print(f"platform      : {platform.platform()}")
    print(f"output        : {output_dir}")
    print("hardware      : CPU exact combinatorics; A100 is not used at this stage")
    print()

    gates: Dict[str, object] = {}

    # ------------------------------------------------------------------
    # G0: exact O(y^3) regression anchors
    # ------------------------------------------------------------------
    b3 = Fraction(1975, 124848)
    leak3 = Fraction(-12331, 249696)
    d3 = Fraction(7, 32) + 12 * leak3 - 4 * b3
    d3_expected = Fraction(-109151, 249696)

    assert d3 == d3_expected

    gates["G0_o3_exact_arithmetic"] = {
        "b3": str(b3),
        "leak3": str(leak3),
        "d3": str(d3),
        "passed": True,
    }

    print(
        f"G0 PASS: d3 = 7/32 + 12*leak3 - 4*b3 = {d3}",
        flush=True,
    )

    # ------------------------------------------------------------------
    # G1: cubic group
    # ------------------------------------------------------------------
    assert len(ROTATIONS) == 24
    assert len(ROOT_STABILIZER) == 8

    for i in ROOT_STABILIZER:
        assert rooted_transform(ROOT, i) == ROOT

    gates["G1_cubic_group"] = {
        "proper_rotations": len(ROTATIONS),
        "root_stabilizer": len(ROOT_STABILIZER),
        "passed": True,
    }

    print("G1 PASS: 24 proper cubic rotations; rooted stabilizer size 8")

    # ------------------------------------------------------------------
    # G2: local incidence anchors
    # ------------------------------------------------------------------
    root_neighbors = plaquettes_containing_vertices(vertices(ROOT))
    assert len(root_neighbors) == 33

    repeated_root = (ROOT, ROOT, ROOT, ROOT)
    repeated_mask = admissible_sign_mask(repeated_root, ROOT)
    assert repeated_mask.bit_count() == 22

    gates["G2_local_anchors"] = {
        "plaquettes_touching_root_by_site": len(root_neighbors),
        "repeated_root_triality_patterns": repeated_mask.bit_count(),
        "passed": True,
    }

    print("G2 PASS: local geometry and SU(3) triality synthetic anchors")

    # ------------------------------------------------------------------
    # G3: enumerate all rooted connected order-4 operator multisets
    # ------------------------------------------------------------------
    support_classes, depth_counts = enumerate_connected_multisets(order=4)

    expected_depth_counts = [1, 6, 156, 5082, 182440]
    assert depth_counts == expected_depth_counts
    assert len(support_classes) == 182440

    gates["G3_support_enumeration"] = {
        "depth_counts": depth_counts,
        "expected": expected_depth_counts,
        "passed": True,
    }

    print(
        "G3 PASS: rooted connected multiset counts "
        + " -> ".join(f"{x:,}" for x in depth_counts)
    )

    # Freeze every support class.
    support_records = [
        [json_plaquette(p) for p in ms]
        for ms in sorted(support_classes)
    ]

    support_path = output_dir / "y4_connected_supports.json.gz"
    support_sha = write_json_gz(
        support_path,
        {
            "meta": {
                "version": VERSION,
                "root": json_plaquette(ROOT),
                "scope": (
                    "rooted site-connected multisets of four perturbation "
                    "plaquettes, repetitions allowed"
                ),
                "proper_rotation_quotient": True,
                "reflection_quotient": False,
            },
            "supports": support_records,
        },
    )

    # ------------------------------------------------------------------
    # G4: outputs + exact triality filter
    # ------------------------------------------------------------------
    candidate_output_pairs = 0
    raw_survivors = 0
    canonical_survivors: Dict[
        Tuple[Tuple[Plaquette, ...], Plaquette],
        int,
    ] = {}

    process_started = time.time()

    for index, ms in enumerate(support_classes, start=1):
        for output in candidate_outputs(ms):
            candidate_output_pairs += 1
            sign_mask = admissible_sign_mask(ms, output)

            if sign_mask:
                raw_survivors += 1
                key = canonical_support_output(ms, output)
                canonical_survivors[key] = (
                    canonical_survivors.get(key, 0) | sign_mask
                )

        if index % 20000 == 0:
            print(
                f"[triality] supports={index:,}/{len(support_classes):,} "
                f"canonical_survivors={len(canonical_survivors):,} "
                f"elapsed={time.time() - process_started:.2f}s",
                flush=True,
            )

    assert candidate_output_pairs == 895524
    assert raw_survivors == 449
    assert len(canonical_survivors) == 449

    gates["G4_triality_filter"] = {
        "candidate_support_output_pairs": candidate_output_pairs,
        "raw_survivors": raw_survivors,
        "canonical_survivors": len(canonical_survivors),
        "passed": True,
    }

    print(
        f"G4 PASS: {candidate_output_pairs:,} candidate output pairs -> "
        f"{len(canonical_survivors):,} exact triality survivors"
    )

    # ------------------------------------------------------------------
    # Build detailed support/output survivor manifest.
    # ------------------------------------------------------------------
    survivor_records: List[dict] = []
    distinct_hist = Counter()
    sign_hist = Counter()
    order_multiplicity_hist = Counter()
    contact_hist = Counter()
    raw_ordered_word_count = 0

    for (ms, output), sign_mask in sorted(canonical_survivors.items()):
        distinct_count = len(set(ms))
        word_mult = ordered_word_multiplicity(ms)
        raw_ordered_word_count += word_mult

        has_link, has_site_only = contact_flags(
            (ROOT,) + tuple(ms) + (output,)
        )

        distinct_hist[distinct_count] += 1
        sign_hist[sign_mask.bit_count()] += 1
        order_multiplicity_hist[word_mult] += 1
        contact_hist[(has_link, has_site_only)] += 1

        all_plaqs = (ROOT,) + tuple(ms) + (output,)

        survivor_records.append(
            {
                "class_id": class_id(ms, output),
                "root": json_plaquette(ROOT),
                "operator_multiset": [json_plaquette(p) for p in ms],
                "output": json_plaquette(output),
                "output_plane": PLANE_NAME[(output[3], output[4])],
                "distinct_operator_plaquettes": distinct_count,
                "ordered_word_multiplicity": word_mult,
                "triality_sign_assignment_count": sign_mask.bit_count(),
                "triality_sign_assignments": signed_patterns_from_mask(sign_mask),
                "support_extent": support_extent(all_plaqs),
                "support_vertices": len(
                    set().union(*(vertex_set(p) for p in all_plaqs))
                ),
                "support_links": len(
                    set().union(*(link_set(p) for p in all_plaqs))
                ),
                "has_link_sharing_contact": has_link,
                "has_site_only_corner_contact": has_site_only,
            }
        )

    assert raw_ordered_word_count == 4296
    assert dict(sorted(distinct_hist.items())) == {
        1: 6,
        2: 170,
        3: 271,
        4: 2,
    }
    assert dict(sorted(sign_hist.items())) == {
        2: 2,
        4: 10,
        8: 416,
        12: 20,
        22: 1,
    }
    assert dict(sorted(order_multiplicity_hist.items())) == {
        1: 6,
        4: 15,
        6: 155,
        12: 271,
        24: 2,
    }
    assert dict(sorted(contact_hist.items())) == {
        (False, False): 1,
        (False, True): 198,
        (True, False): 46,
        (True, True): 204,
    }

    gates["G5_survivor_classification"] = {
        "distinct_operator_histogram": dict(sorted(distinct_hist.items())),
        "sign_assignment_histogram": dict(sorted(sign_hist.items())),
        "ordered_word_multiplicity_histogram": dict(
            sorted(order_multiplicity_hist.items())
        ),
        "contact_histogram": {
            f"link={k[0]},site_only={k[1]}": v
            for k, v in sorted(contact_hist.items())
        },
        "raw_ordered_words": raw_ordered_word_count,
        "passed": True,
    }

    print(
        "G5 PASS: survivor classification; "
        f"raw ordered words={raw_ordered_word_count:,}"
    )

    survivor_path = output_dir / "y4_triality_survivors.json.gz"
    survivor_sha = write_json_gz(
        survivor_path,
        {
            "meta": {
                "version": VERSION,
                "root": json_plaquette(ROOT),
                "filter": (
                    "necessary SU(3) link triality condition only; "
                    "not a sufficient nonzero Haar criterion"
                ),
                "sign_order": [
                    "ket",
                    "operator_1",
                    "operator_2",
                    "operator_3",
                    "operator_4",
                    "bra",
                ],
                "bra_is_conjugated": True,
            },
            "classes": survivor_records,
        },
    )

    # ------------------------------------------------------------------
    # G6: expand and symmetry-quotient ordered transitions
    # ------------------------------------------------------------------
    ordered_transitions: Dict[
        Tuple[Tuple[Plaquette, ...], Plaquette],
        int,
    ] = {}

    for (ms, output), _support_mask in canonical_survivors.items():
        for word in set(itertools.permutations(ms)):
            key = canonical_ordered_transition(word, output)

            # Recompute exact sign mask in the ordered column convention.
            word_mask = admissible_sign_mask(word, output)
            assert word_mask != 0

            ordered_transitions[key] = (
                ordered_transitions.get(key, 0) | word_mask
            )

    assert len(ordered_transitions) == 4221

    ordered_records: List[dict] = []

    for index, ((word, output), sign_mask) in enumerate(
        sorted(ordered_transitions.items()),
        start=1,
    ):
        ordered_records.append(
            {
                "ordered_id": f"W4-{index:05d}",
                "root": json_plaquette(ROOT),
                "ordered_insertions": [json_plaquette(p) for p in word],
                "output": json_plaquette(output),
                "triality_sign_assignment_count": sign_mask.bit_count(),
                "triality_sign_assignments": signed_patterns_from_mask(sign_mask),
            }
        )

    gates["G6_ordered_words"] = {
        "raw_words_before_rooted_symmetry": raw_ordered_word_count,
        "canonical_ordered_transitions": len(ordered_transitions),
        "passed": True,
    }

    print(
        f"G6 PASS: {raw_ordered_word_count:,} raw words -> "
        f"{len(ordered_transitions):,} rooted symmetry classes"
    )

    ordered_path = output_dir / "y4_ordered_transition_words.json.gz"
    ordered_sha = write_json_gz(
        ordered_path,
        {
            "meta": {
                "version": VERSION,
                "root": json_plaquette(ROOT),
                "word_length": 4,
                "proper_rotation_quotient": True,
                "reflection_quotient": False,
                "scope": (
                    "connected triality-admissible candidate words; "
                    "intermediate SU(3) channels and denominators not yet attached"
                ),
            },
            "words": ordered_records,
        },
    )

    # ------------------------------------------------------------------
    # G7: round trip and final summary
    # ------------------------------------------------------------------
    assert len(read_json_gz(support_path)["supports"]) == 182440
    assert len(read_json_gz(survivor_path)["classes"]) == 449
    assert len(read_json_gz(ordered_path)["words"]) == 4221

    gates["G7_round_trip"] = {
        "supports": 182440,
        "survivors": 449,
        "ordered_transitions": 4221,
        "passed": True,
    }

    elapsed = time.time() - started

    summary = {
        "meta": {
            "version": VERSION,
            "date": "2026-06-13",
            "python": sys.version,
            "platform": platform.platform(),
            "walltime_s": elapsed,
            "hardware": "CPU",
            "a100_required": False,
        },
        "scope": {
            "completed": [
                "rooted connected support enumeration",
                "proper cubic symmetry quotient",
                "repeated insertion bookkeeping",
                "candidate external plaquette attachment",
                "necessary exact SU(3) link-triality filter",
                "ordered fourth-order word expansion",
            ],
            "not_completed": [
                "SU(3) Haar contractions",
                "intermediate representation-channel enumeration",
                "electric-energy denominators",
                "des-Cloizeaux folded/subtraction terms",
                "exact fourth-order rational weights",
                "H4 flat-band commutator",
            ],
        },
        "counts": {
            "connected_support_multisets": 182440,
            "candidate_support_output_pairs": 895524,
            "triality_survivor_classes": 449,
            "raw_ordered_words": 4296,
            "canonical_ordered_transition_words": 4221,
            "classes_with_site_only_corner_contact": (
                contact_hist[(False, True)] + contact_hist[(True, True)]
            ),
            "classes_without_site_only_corner_contact": (
                contact_hist[(False, False)] + contact_hist[(True, False)]
            ),
            "classes_with_any_link_sharing": (
                contact_hist[(True, False)] + contact_hist[(True, True)]
            ),
            "classes_with_no_link_sharing": (
                contact_hist[(False, False)] + contact_hist[(False, True)]
            ),
            "pure_site_only_classes": contact_hist[(False, True)],
        },
        "histograms": {
            "distinct_operator_plaquettes": dict(sorted(distinct_hist.items())),
            "triality_sign_assignments": dict(sorted(sign_hist.items())),
            "ordered_word_multiplicity": dict(
                sorted(order_multiplicity_hist.items())
            ),
        },
        "gates": gates,
        "files": {
            support_path.name: {
                "sha256": support_sha,
                "records": 182440,
            },
            survivor_path.name: {
                "sha256": survivor_sha,
                "records": 449,
            },
            ordered_path.name: {
                "sha256": ordered_sha,
                "records": 4221,
            },
        },
        "next_stage": (
            "Attach exact SU(3) representation channels and des-Cloizeaux "
            "energy-denominator words to the 4,221 ordered transition classes."
        ),
        "passed": True,
    }

    summary_path = output_dir / "y4_stage0_summary.json"
    summary_sha = write_json(summary_path, summary)

    print("G7 PASS: gzip JSON round-trip and exact record counts")
    print()
    print("SUMMARY")
    print(json.dumps(summary["counts"], indent=2, sort_keys=True))
    print()
    print(f"SUPPORTS : {support_path}")
    print(f"SURVIVORS: {survivor_path}")
    print(f"WORDS    : {ordered_path}")
    print(f"SUMMARY  : {summary_path}")
    print(f"SUMMARY SHA256: {summary_sha}")
    print(f"WALLTIME : {elapsed:.2f} s")
    print("ALL STAGE-0 GATES PASS")
    print()
    print(
        "NEXT: exact SU(3) channel/denominator attachment. "
        "Use the A100 only after the contraction tensor dimensions are known."
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="O(y^4) connected geometry and SU(3) triality manifest"
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=default_output_dir(),
    )
    args, _unknown = parser.parse_known_args()
    main(args.output_dir)


SU(3) O(y^4) STAGE-0 CONNECTED GEOMETRY / TRIALITY MANIFEST
version       : 2026-06-13-stage0-v1
python        : 3.12.13
platform      : Linux-6.6.122+-x86_64-with-glibc2.35
output        : /content/Y4_STAGE0
hardware      : CPU exact combinatorics; A100 is not used at this stage

G0 PASS: d3 = 7/32 + 12*leak3 - 4*b3 = -109151/249696
G1 PASS: 24 proper cubic rotations; rooted stabilizer size 8
G2 PASS: local geometry and SU(3) triality synthetic anchors
[enumerate] depth=1 classes=6 elapsed=0.00s
[enumerate] depth=2 classes=156 elapsed=0.00s
[enumerate] depth=3 classes=5,082 elapsed=0.19s
[enumerate] depth=4 classes=182,440 elapsed=11.54s
G3 PASS: rooted connected multiset counts 1 -> 6 -> 156 -> 5,082 -> 182,440
[triality] supports=20,000/182,440 canonical_survivors=60 elapsed=3.66s
[triality] supports=40,000/182,440 canonical_survivors=103 elapsed=5.90s
[triality] supports=60,000/182,440 canonical_survivors=163 elapsed=8.14s
[triality] supports=80,000/182,440 canonical_survivors=209 

In [3]:
import json
import fractions

# SU(3) Link Casimirs above electric vacuum (E_0 = 8/3)
CASIMIRS = {
    '1': 0,
    '3': 4/3,
    '3bar': 4/3,
    '8': 3,
    '6': 10/3,
    '6bar': 10/3
}

def compute_energy_penalty(link_states):
    """
    Computes the exact rational energy penalty \Delta E_i for an intermediate state.
    link_states: list of string channel identifiers for the 16 active links.
    """
    total_energy = sum([CASIMIRS[state] for state in link_states])
    return fractions.Fraction(total_energy).limit_denominator()

def process_geometry_shard(shard_data):
    """
    Ingests a chunk of pure_site_only_classes.
    Applies the bare-link topological nulling and calculates denominator constraints.
    """
    compiled_tensors = []
    local_denominators = []

    for geometry in shard_data:
        # Step 1: Topological Bare-Link Check
        # (Insert your specific graph traversal logic here to check for unclosed nontrivial links)
        if topology_fails_bare_link_check(geometry):
            continue

        # Step 2: Denominator Extraction
        # Evaluate the 3 intermediate states between the 4 plaquette operators
        penalty_1 = compute_energy_penalty(geometry['state_1_links'])
        penalty_2 = compute_energy_penalty(geometry['state_2_links'])
        penalty_3 = compute_energy_penalty(geometry['state_3_links'])

        word_scalar = fractions.Fraction(1, (penalty_1 * penalty_2 * penalty_3))
        local_denominators.append(word_scalar.denominator)

        # Step 3: Tensor Shape Definition
        compiled_tensors.append({
            'class_id': geometry['id'],
            'scalar_numerator': word_scalar.numerator,
            'scalar_denominator': word_scalar.denominator,
            'tensor_shape': [3, 3, 3, 3, 3, 3, 3, 3], # 8-rank for 4-plaquette corner
            'contraction_indices': geometry['index_map']
        })

    return compiled_tensors, local_denominators

<>:16: SyntaxWarning: invalid escape sequence '\D'
<>:16: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_382/44803044.py:16: SyntaxWarning: invalid escape sequence '\D'
  Computes the exact rational energy penalty \Delta E_i for an intermediate state.


In [4]:
import json
import fractions

# SU(3) Link Casimirs above electric vacuum (E_0 = 8/3)
CASIMIRS = {
    '1': 0,
    '3': 4/3,
    '3bar': 4/3,
    '8': 3,
    '6': 10/3,
    '6bar': 10/3
}

def compute_energy_penalty(link_states):
    r"""
    Computes the exact rational energy penalty \Delta E_i for an intermediate state.
    link_states: list of string channel identifiers for the 16 active links.
    """
    total_energy = sum([CASIMIRS[state] for state in link_states])
    return fractions.Fraction(total_energy).limit_denominator()

def process_geometry_shard(shard_data):
    """
    Ingests a chunk of pure_site_only_classes.
    Applies the bare-link topological nulling and calculates denominator constraints.
    """
    compiled_tensors = []
    local_denominators = []

    for geometry in shard_data:
        # Step 1: Topological Bare-Link Check
        # (Insert specific graph traversal logic here to check for unclosed nontrivial links)
        if topology_fails_bare_link_check(geometry):
            continue

        # Step 2: Denominator Extraction
        # Evaluate the 3 intermediate states between the 4 plaquette operators
        penalty_1 = compute_energy_penalty(geometry['state_1_links'])
        penalty_2 = compute_energy_penalty(geometry['state_2_links'])
        penalty_3 = compute_energy_penalty(geometry['state_3_links'])

        word_scalar = fractions.Fraction(1, (penalty_1 * penalty_2 * penalty_3))
        local_denominators.append(word_scalar.denominator)

        # Step 3: Tensor Shape Definition
        compiled_tensors.append({
            'class_id': geometry['id'],
            'scalar_numerator': word_scalar.numerator,
            'scalar_denominator': word_scalar.denominator,
            'tensor_shape': [3, 3, 3, 3, 3, 3, 3, 3], # 8-rank for 4-plaquette corner
            'contraction_indices': geometry['index_map']
        })

    return compiled_tensors, local_denominators

In [5]:
import json
import fractions

# SU(3) Link Casimirs above electric vacuum (E_0 = 8/3)
CASIMIRS = {
    '1': fractions.Fraction(0, 1),
    '3': fractions.Fraction(4, 3),
    '3bar': fractions.Fraction(4, 3),
    '8': fractions.Fraction(3, 1),
    '6': fractions.Fraction(10, 3),
    '6bar': fractions.Fraction(10, 3)
}

def compute_energy_penalty(link_states):
    r"""
    Computes the exact rational energy penalty \Delta E_i for an intermediate state.
    link_states: list of string channel identifiers for the 16 active links.
    """
    return sum((CASIMIRS[state] for state in link_states), fractions.Fraction(0, 1))

def process_geometry_shard(shard_data, topology_fails_bare_link_check):
    """
    Ingests a chunk of pure_site_only_classes.
    Applies the bare-link topological nulling and calculates denominator constraints.
    """
    compiled_tensors = []
    local_denominators = []

    for geometry in shard_data:
        # Step 1: Topological Bare-Link Check
        if topology_fails_bare_link_check(geometry):
            continue

        # Step 2: Denominator Extraction
        penalty_1 = compute_energy_penalty(geometry['state_1_links'])
        penalty_2 = compute_energy_penalty(geometry['state_2_links'])
        penalty_3 = compute_energy_penalty(geometry['state_3_links'])

        # Guard against zero division if a pure vacuum state somehow enters the path
        if penalty_1 == 0 or penalty_2 == 0 or penalty_3 == 0:
            continue

        word_scalar = fractions.Fraction(1, (penalty_1 * penalty_2 * penalty_3))
        local_denominators.append(word_scalar.denominator)

        # Step 3: Tensor Shape Definition
        compiled_tensors.append({
            'class_id': geometry['id'],
            'scalar_numerator': word_scalar.numerator,
            'scalar_denominator': word_scalar.denominator,
            'tensor_shape': [3, 3, 3, 3, 3, 3, 3, 3], # 8-rank for 4-plaquette corner
            'contraction_indices': geometry['index_map']
        })

    return compiled_tensors, local_denominators


if __name__ == "__main__":
    # --- MOCK EXECUTION TO VERIFY SYNTAX AND RUNTIME ---
    mock_shard = [{
        'id': 'word_001',
        'state_1_links': ['3', '3bar'] + ['1']*14,
        'state_2_links': ['8'] + ['1']*15,
        'state_3_links': ['3', '3bar'] + ['1']*14,
        'index_map': [0, 1, 2, 3, 4, 5, 6, 7]
    }]

    # Stub for your exact graph traversal logic
    def mock_topology_check(geo):
        return False

    tensors, denoms = process_geometry_shard(mock_shard, mock_topology_check)
    print(f"Compiled Tensors: {json.dumps(tensors, indent=2)}")
    print(f"Local Denominators: {denoms}")

Compiled Tensors: [
  {
    "class_id": "word_001",
    "scalar_numerator": 3,
    "scalar_denominator": 64,
    "tensor_shape": [
      3,
      3,
      3,
      3,
      3,
      3,
      3,
      3
    ],
    "contraction_indices": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7
    ]
  }
]
Local Denominators: [64]


In [6]:
#!/usr/bin/env python3
"""
y4_stage2_exact_haar_library.py
===============================

Exact local SU(3) Haar-projector compiler for the O(y^4) one-flux program.

RUN AFTER STAGE 1
-----------------
    %run /content/y4_stage2_exact_haar_library.py

HARDWARE
--------
Use a standard Colab CPU runtime. The A100 is not used.

INPUT
-----
    /content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz

OUTPUT
------
    /content/Y4_STAGE2/su3_local_haar_projectors.json
    /content/Y4_STAGE2/y4_link_tensor_cards.json.gz
    /content/Y4_STAGE2/y4_stage2_summary.json

WHAT THIS STAGE COMPUTES
------------------------
For every local link moment family surviving Stage 1, this script constructs
the exact Haar projector

    ∫ dU  R(U) = sum_{a,b} |T_a> (G^{-1})_{ab} <T_b|,

where {T_a} is an explicit invariant tensor basis and
G_ab = <T_a,T_b>.

The five families are:

    (1,1)  delta basis, invariant dimension 1
    (2,2)  permutation-delta basis, invariant dimension 2
    (3,0)  epsilon basis, invariant dimension 1
    (3,3)  permutation-delta basis, invariant dimension 6
    (6,0)  double-epsilon basis, invariant dimension 5

The script also maps all 182 distinct Stage-1 link token signatures to exact
tensor cards in the physical event order:

    ket, insertion_1, insertion_2, insertion_3, insertion_4, bra.

IMPORTANT SCOPE
---------------
This stage computes the exact LOCAL Haar tensors. It does not yet contract the
plaquette trace-index network across links and does not compute final O(y^4)
weights or des-Cloizeaux folded terms.

All arithmetic is exact Python Fraction/integer arithmetic.
"""

from __future__ import annotations

import argparse
import gzip
import hashlib
import itertools
import json
import platform
import sys
import time
from collections import Counter, defaultdict
from fractions import Fraction
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

VERSION = "2026-06-13-stage2-v1"
N = 3
EVENT_NAMES = [
    "ket",
    "insertion_1",
    "insertion_2",
    "insertion_3",
    "insertion_4",
    "bra",
]

TokenSignature = Tuple[int, int, int, int, int, int]
Matrix = List[List[Fraction]]


def default_input() -> Path:
    candidates = [
        Path("/content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz"),
        Path.cwd() / "Y4_STAGE1" / "y4_channel_denominator_manifest.json.gz",
        Path("/mnt/data/Y4_STAGE1_TEST2/y4_channel_denominator_manifest.json.gz"),
        Path("/mnt/data/Y4_STAGE1_TEST/y4_channel_denominator_manifest.json.gz"),
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def default_output_dir() -> Path:
    if Path("/content").exists():
        return Path("/content/Y4_STAGE2")
    return Path.cwd() / "Y4_STAGE2"


# ---------------------------------------------------------------------------
# Exact matrix algebra
# ---------------------------------------------------------------------------

def identity_matrix(n: int) -> Matrix:
    return [
        [Fraction(int(i == j), 1) for j in range(n)]
        for i in range(n)
    ]


def matrix_inverse(matrix: Sequence[Sequence[int | Fraction]]) -> Matrix:
    n = len(matrix)
    assert n > 0 and all(len(row) == n for row in matrix)

    aug: Matrix = [
        [Fraction(x) for x in matrix[i]] + identity_matrix(n)[i]
        for i in range(n)
    ]

    for col in range(n):
        pivot = next((r for r in range(col, n) if aug[r][col] != 0), None)
        assert pivot is not None, "singular matrix"
        aug[col], aug[pivot] = aug[pivot], aug[col]

        z = aug[col][col]
        aug[col] = [x / z for x in aug[col]]

        for row in range(n):
            if row == col:
                continue
            z = aug[row][col]
            if z:
                aug[row] = [
                    aug[row][j] - z * aug[col][j]
                    for j in range(2 * n)
                ]

    return [row[n:] for row in aug]


def matrix_multiply(a: Matrix, b: Matrix) -> Matrix:
    assert a and b and len(a[0]) == len(b)
    return [
        [
            sum(a[i][k] * b[k][j] for k in range(len(b)))
            for j in range(len(b[0]))
        ]
        for i in range(len(a))
    ]


def matrix_rank(matrix: Sequence[Sequence[int | Fraction]]) -> int:
    if not matrix:
        return 0

    a: Matrix = [[Fraction(x) for x in row] for row in matrix]
    m, n = len(a), len(a[0])
    rank = 0

    for col in range(n):
        pivot = next((r for r in range(rank, m) if a[r][col] != 0), None)
        if pivot is None:
            continue

        a[rank], a[pivot] = a[pivot], a[rank]
        z = a[rank][col]
        a[rank] = [x / z for x in a[rank]]

        for row in range(m):
            if row != rank and a[row][col]:
                z = a[row][col]
                a[row] = [
                    a[row][j] - z * a[rank][j]
                    for j in range(n)
                ]

        rank += 1
        if rank == m:
            break

    return rank


def assert_inverse(gram: Matrix, inverse: Matrix) -> None:
    assert matrix_multiply(gram, inverse) == identity_matrix(len(gram))
    assert matrix_multiply(inverse, gram) == identity_matrix(len(gram))


def frac_string(x: Fraction) -> str:
    return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"


def matrix_strings(matrix: Matrix) -> List[List[str]]:
    return [[frac_string(x) for x in row] for row in matrix]


# ---------------------------------------------------------------------------
# Permutations and balanced U/U* invariant tensors
# ---------------------------------------------------------------------------

Permutation = Tuple[int, ...]


def inverse_permutation(p: Permutation) -> Permutation:
    out = [0] * len(p)
    for i, value in enumerate(p):
        out[value] = i
    return tuple(out)


def compose_permutations(p: Permutation, q: Permutation) -> Permutation:
    """p after q."""
    return tuple(p[q[i]] for i in range(len(p)))


def permutation_cycles(p: Permutation) -> int:
    seen = [False] * len(p)
    count = 0
    for start in range(len(p)):
        if seen[start]:
            continue
        count += 1
        j = start
        while not seen[j]:
            seen[j] = True
            j = p[j]
    return count


def balanced_family(k: int) -> dict:
    permutations = list(itertools.permutations(range(k)))
    gram: Matrix = []

    for sigma in permutations:
        inv_sigma = inverse_permutation(sigma)
        row: List[Fraction] = []
        for tau in permutations:
            relative = compose_permutations(inv_sigma, tau)
            row.append(Fraction(N ** permutation_cycles(relative), 1))
        gram.append(row)

    inverse = matrix_inverse(gram)
    assert_inverse(gram, inverse)

    basis = [
        {
            "basis_index": i,
            "type": "delta_pairing",
            "fund_to_antifund_permutation": list(p),
            "formula": "product_r delta(fund[r], antifund[perm[r]])",
        }
        for i, p in enumerate(permutations)
    ]

    return {
        "family": f"({k},{k})",
        "n_fund": k,
        "n_antifund": k,
        "degree": 2 * k,
        "basis_type": "permutation_delta",
        "invariant_dimension": len(permutations),
        "basis": basis,
        "gram": gram,
        "gram_inverse": inverse,
    }


# ---------------------------------------------------------------------------
# Epsilon tensors and unbalanced SU(3) invariants
# ---------------------------------------------------------------------------

def epsilon3(a: int, b: int, c: int) -> int:
    if len({a, b, c}) < 3:
        return 0
    inversions = int(a > b) + int(a > c) + int(b > c)
    return -1 if inversions % 2 else +1


def epsilon3_family(n_fund: int, n_antifund: int) -> dict:
    assert sorted((n_fund, n_antifund)) == [0, 3]
    gram: Matrix = [[Fraction(6, 1)]]
    inverse: Matrix = [[Fraction(1, 6)]]
    assert_inverse(gram, inverse)

    occupied_type = "fund" if n_fund == 3 else "antifund"

    return {
        "family": f"({n_fund},{n_antifund})",
        "n_fund": n_fund,
        "n_antifund": n_antifund,
        "degree": 3,
        "basis_type": "epsilon",
        "invariant_dimension": 1,
        "basis": [
            {
                "basis_index": 0,
                "type": "epsilon",
                "slots": [0, 1, 2],
                "slot_species": occupied_type,
                "formula": "epsilon(index[0],index[1],index[2])",
            }
        ],
        "gram": gram,
        "gram_inverse": inverse,
    }


def epsilon_partition_candidates() -> List[Tuple[Tuple[int, ...], Tuple[int, ...]]]:
    """Ten unordered partitions of six slots into two triples."""
    universe = set(range(6))
    out = []
    for pair in itertools.combinations(range(1, 6), 2):
        first = tuple(sorted((0,) + pair))
        second = tuple(sorted(universe - set(first)))
        out.append((first, second))
    assert len(out) == 10
    return out


def epsilon_partition_value(
    partition: Tuple[Tuple[int, ...], Tuple[int, ...]],
    indices: Sequence[int],
) -> int:
    a, b = partition
    return epsilon3(*(indices[i] for i in a)) * epsilon3(
        *(indices[i] for i in b)
    )


def epsilon6_family(n_fund: int, n_antifund: int) -> dict:
    assert sorted((n_fund, n_antifund)) == [0, 6]

    candidates = epsilon_partition_candidates()
    color_tuples = list(itertools.product(range(N), repeat=6))
    candidate_vectors = [
        [epsilon_partition_value(partition, x) for x in color_tuples]
        for partition in candidates
    ]

    full_gram_int = [
        [
            sum(
                candidate_vectors[i][r] * candidate_vectors[j][r]
                for r in range(len(color_tuples))
            )
            for j in range(len(candidates))
        ]
        for i in range(len(candidates))
    ]

    assert matrix_rank(full_gram_int) == 5

    selected: List[int] = []
    for candidate_index in range(len(candidates)):
        trial = selected + [candidate_index]
        sub = [
            [full_gram_int[i][j] for j in trial]
            for i in trial
        ]
        if matrix_rank(sub) > len(selected):
            selected.append(candidate_index)
        if len(selected) == 5:
            break

    assert selected == [0, 1, 2, 4, 5]

    gram: Matrix = [
        [Fraction(full_gram_int[i][j], 1) for j in selected]
        for i in selected
    ]
    inverse = matrix_inverse(gram)
    assert_inverse(gram, inverse)

    occupied_type = "fund" if n_fund == 6 else "antifund"

    basis = []
    for basis_index, candidate_index in enumerate(selected):
        first, second = candidates[candidate_index]
        basis.append(
            {
                "basis_index": basis_index,
                "candidate_index": candidate_index,
                "type": "double_epsilon",
                "first_triple": list(first),
                "second_triple": list(second),
                "slot_species": occupied_type,
                "formula": "epsilon(first_triple)*epsilon(second_triple)",
            }
        )

    return {
        "family": f"({n_fund},{n_antifund})",
        "n_fund": n_fund,
        "n_antifund": n_antifund,
        "degree": 6,
        "basis_type": "double_epsilon",
        "invariant_dimension": 5,
        "all_candidate_partitions": [
            [list(a), list(b)] for a, b in candidates
        ],
        "candidate_gram": [
            [int(x) for x in row] for row in full_gram_int
        ],
        "candidate_rank": 5,
        "selected_candidate_indices": selected,
        "basis": basis,
        "gram": gram,
        "gram_inverse": inverse,
    }


# ---------------------------------------------------------------------------
# Invariant basis evaluation and exact Haar projector entries
# ---------------------------------------------------------------------------

def basis_value(family: dict, basis_index: int, multi_index: Sequence[int]) -> int:
    nf = family["n_fund"]
    na = family["n_antifund"]
    assert len(multi_index) == nf + na
    assert all(0 <= x < N for x in multi_index)

    basis = family["basis"][basis_index]
    kind = family["basis_type"]

    if kind == "permutation_delta":
        fund = multi_index[:nf]
        anti = multi_index[nf:]
        perm = basis["fund_to_antifund_permutation"]
        value = 1
        for r in range(nf):
            value *= int(fund[r] == anti[perm[r]])
        return value

    if kind == "epsilon":
        return epsilon3(*multi_index)

    if kind == "double_epsilon":
        first = basis["first_triple"]
        second = basis["second_triple"]
        return epsilon3(*(multi_index[i] for i in first)) * epsilon3(
            *(multi_index[i] for i in second)
        )

    raise ValueError(kind)


def projector_entry(
    family: dict,
    row_multi_index: Sequence[int],
    column_multi_index: Sequence[int],
) -> Fraction:
    dim = family["invariant_dimension"]
    left = [
        Fraction(basis_value(family, a, row_multi_index), 1)
        for a in range(dim)
    ]
    right = [
        Fraction(basis_value(family, b, column_multi_index), 1)
        for b in range(dim)
    ]
    inv = family["gram_inverse"]

    return sum(
        left[a] * inv[a][b] * right[b]
        for a in range(dim)
        for b in range(dim)
    )


def projector_trace(family: dict) -> Fraction:
    degree = family["degree"]
    return sum(
        projector_entry(family, x, x)
        for x in itertools.product(range(N), repeat=degree)
    )


# ---------------------------------------------------------------------------
# Stage-1 token-card mapping
# ---------------------------------------------------------------------------

def canonical_family_key(n_fund: int, n_antifund: int) -> Tuple[int, int]:
    return tuple(sorted((n_fund, n_antifund)))


def actual_family_name(n_fund: int, n_antifund: int) -> str:
    return f"({n_fund},{n_antifund})"


def basis_card_for_signature(
    signature: TokenSignature,
    family: dict,
    occurrence_count: int,
) -> dict:
    fund_positions = [i for i, x in enumerate(signature) if x == +1]
    antifund_positions = [i for i, x in enumerate(signature) if x == -1]

    basis_cards = []
    kind = family["basis_type"]

    if kind == "permutation_delta":
        for basis in family["basis"]:
            perm = basis["fund_to_antifund_permutation"]
            pairs = [
                [fund_positions[r], antifund_positions[perm[r]]]
                for r in range(len(fund_positions))
            ]
            basis_cards.append(
                {
                    "basis_index": basis["basis_index"],
                    "type": "delta_pairing",
                    "event_position_pairs": pairs,
                    "event_name_pairs": [
                        [EVENT_NAMES[a], EVENT_NAMES[b]]
                        for a, b in pairs
                    ],
                }
            )

    elif kind == "epsilon":
        positions = fund_positions if fund_positions else antifund_positions
        basis_cards.append(
            {
                "basis_index": 0,
                "type": "epsilon",
                "event_positions": positions,
                "event_names": [EVENT_NAMES[i] for i in positions],
                "species": "fund" if fund_positions else "antifund",
            }
        )

    elif kind == "double_epsilon":
        positions = fund_positions if fund_positions else antifund_positions
        species = "fund" if fund_positions else "antifund"
        for basis in family["basis"]:
            first = [positions[i] for i in basis["first_triple"]]
            second = [positions[i] for i in basis["second_triple"]]
            basis_cards.append(
                {
                    "basis_index": basis["basis_index"],
                    "type": "double_epsilon",
                    "first_event_positions": first,
                    "second_event_positions": second,
                    "first_event_names": [EVENT_NAMES[i] for i in first],
                    "second_event_names": [EVENT_NAMES[i] for i in second],
                    "species": species,
                }
            )
    else:
        raise ValueError(kind)

    return {
        "token_signature": list(signature),
        "event_tokens": {
            EVENT_NAMES[i]: signature[i] for i in range(6)
        },
        "c_orbit_representative_occurrence_count": occurrence_count,
        "n_fund": len(fund_positions),
        "n_antifund": len(antifund_positions),
        "actual_family": actual_family_name(
            len(fund_positions), len(antifund_positions)
        ),
        "canonical_family": f"({canonical_family_key(len(fund_positions), len(antifund_positions))[0]},"
                            f"{canonical_family_key(len(fund_positions), len(antifund_positions))[1]})",
        "fund_event_positions": fund_positions,
        "antifund_event_positions": antifund_positions,
        "invariant_dimension": family["invariant_dimension"],
        "basis_type": kind,
        "basis_in_event_order": basis_cards,
        "gram_inverse": matrix_strings(family["gram_inverse"]),
        "integral_formula": (
            "Integral = sum_ab T_a(row color indices) "
            "(G^-1)_ab T_b(column color indices)"
        ),
    }


# ---------------------------------------------------------------------------
# I/O
# ---------------------------------------------------------------------------

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, obj: object) -> str:
    raw = json.dumps(obj, indent=2, sort_keys=True, allow_nan=False).encode("utf-8")
    path.write_bytes(raw)
    return hashlib.sha256(raw).hexdigest()


def write_json_gz(path: Path, obj: object) -> str:
    raw = json.dumps(obj, separators=(",", ":"), sort_keys=True, allow_nan=False).encode("utf-8")
    with gzip.GzipFile(filename=str(path), mode="wb", compresslevel=9, mtime=0) as handle:
        handle.write(raw)
    return sha256_file(path)


def read_json_gz(path: Path) -> object:
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return json.load(handle)


def serialize_family(family: dict) -> dict:
    out = dict(family)
    out["gram"] = matrix_strings(family["gram"])
    out["gram_inverse"] = matrix_strings(family["gram_inverse"])
    return out


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main(input_path: Path, output_dir: Path) -> None:
    started = time.time()
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("SU(3) O(y^4) STAGE-2 EXACT LOCAL HAAR PROJECTOR LIBRARY")
    print("=" * 100)
    print(f"version  : {VERSION}")
    print(f"input    : {input_path}")
    print(f"output   : {output_dir}")
    print("hardware : standard Colab CPU; A100 is not used")
    print()

    assert input_path.exists(), f"Stage-1 input not found: {input_path}"
    input_sha = sha256_file(input_path)
    stage1 = read_json_gz(input_path)
    words = stage1["words"]
    assert len(words) == 4221

    gates: Dict[str, object] = {}

    # ------------------------------------------------------------------
    # G0: construct exact invariant families.
    # ------------------------------------------------------------------
    families: Dict[Tuple[int, int], dict] = {
        (1, 1): balanced_family(1),
        (2, 2): balanced_family(2),
        (0, 3): epsilon3_family(0, 3),
        (3, 0): epsilon3_family(3, 0),
        (3, 3): balanced_family(3),
        (0, 6): epsilon6_family(0, 6),
        (6, 0): epsilon6_family(6, 0),
    }

    expected_dimensions = {
        (1, 1): 1,
        (2, 2): 2,
        (0, 3): 1,
        (3, 0): 1,
        (3, 3): 6,
        (0, 6): 5,
        (6, 0): 5,
    }
    assert {
        key: family["invariant_dimension"]
        for key, family in families.items()
    } == expected_dimensions

    gates["G0_invariant_dimensions"] = {
        str(key): value for key, value in expected_dimensions.items()
    }
    gates["G0_invariant_dimensions"]["passed"] = True
    print("G0 PASS: exact invariant bases dimensions 1,2,1,6,5")

    # ------------------------------------------------------------------
    # G1: exact Weingarten / epsilon coefficient anchors.
    # ------------------------------------------------------------------
    assert families[(1, 1)]["gram_inverse"] == [[Fraction(1, 3)]]

    assert families[(2, 2)]["gram_inverse"] == [
        [Fraction(1, 8), Fraction(-1, 24)],
        [Fraction(-1, 24), Fraction(1, 8)],
    ]

    wg3 = families[(3, 3)]["gram_inverse"]
    allowed_wg3 = {Fraction(7, 120), Fraction(-1, 40), Fraction(1, 60)}
    assert {x for row in wg3 for x in row} == allowed_wg3

    assert families[(0, 3)]["gram_inverse"] == [[Fraction(1, 6)]]
    assert families[(3, 0)]["gram_inverse"] == [[Fraction(1, 6)]]

    expected_eps6_inverse = [
        [Fraction(1, 12), Fraction(-1, 24), Fraction(1, 24), Fraction(1, 24), Fraction(-1, 24)],
        [Fraction(-1, 24), Fraction(1, 18), Fraction(-1, 36), Fraction(-1, 36), Fraction(1, 72)],
        [Fraction(1, 24), Fraction(-1, 36), Fraction(1, 18), Fraction(1, 72), Fraction(-1, 36)],
        [Fraction(1, 24), Fraction(-1, 36), Fraction(1, 72), Fraction(1, 18), Fraction(-1, 36)],
        [Fraction(-1, 24), Fraction(1, 72), Fraction(-1, 36), Fraction(-1, 36), Fraction(1, 18)],
    ]
    assert families[(0, 6)]["gram_inverse"] == expected_eps6_inverse
    assert families[(6, 0)]["gram_inverse"] == expected_eps6_inverse

    gates["G1_exact_coefficients"] = {
        "Wg_1": "1/3",
        "Wg_2_identity": "1/8",
        "Wg_2_transposition": "-1/24",
        "Wg_3_identity": "7/120",
        "Wg_3_transposition": "-1/40",
        "Wg_3_three_cycle": "1/60",
        "epsilon3": "1/6",
        "epsilon6_selected_basis_inverse": matrix_strings(expected_eps6_inverse),
        "passed": True,
    }
    print("G1 PASS: exact SU(3) Weingarten and epsilon coefficients")

    # ------------------------------------------------------------------
    # G2: exact Haar projector trace/rank checks.
    # ------------------------------------------------------------------
    trace_results = {}
    for key, family in families.items():
        trace = projector_trace(family)
        assert trace == family["invariant_dimension"]
        trace_results[str(key)] = frac_string(trace)

        # Hermiticity spot-check over deterministic color tuples.
        degree = family["degree"]
        tuples = list(itertools.product(range(N), repeat=degree))
        sample = tuples[: min(12, len(tuples))]
        sample += tuples[-min(12, len(tuples)):]
        for x in sample:
            for y in sample:
                assert projector_entry(family, x, y) == projector_entry(family, y, x)

    # Explicit determinant anchors.
    assert projector_entry(
        families[(3, 0)],
        (0, 1, 2),
        (0, 1, 2),
    ) == Fraction(1, 6)
    assert projector_entry(
        families[(3, 0)],
        (0, 1, 2),
        (0, 2, 1),
    ) == Fraction(-1, 6)

    gates["G2_projector_checks"] = {
        "projector_traces": trace_results,
        "epsilon_spot_checks": True,
        "hermiticity_spot_checks": True,
        "passed": True,
    }
    print("G2 PASS: exact projector traces, Hermiticity, determinant anchors")

    # ------------------------------------------------------------------
    # G3: scan every Stage-1 local link signature.
    # ------------------------------------------------------------------
    signature_occurrences: Counter[TokenSignature] = Counter()

    for word in words:
        for orbit in word["orientation_orbits"]:
            assert orbit["exact_singlet_feasible"] is True
            for token_list in orbit["link_token_signatures"]:
                signature = tuple(int(x) for x in token_list)
                assert len(signature) == 6
                assert all(x in (-1, 0, +1) for x in signature)
                signature_occurrences[signature] += 1

    assert len(signature_occurrences) == 182
    assert sum(signature_occurrences.values()) == 187632

    unique_family_hist = Counter()
    occurrence_family_hist = Counter()
    cards = []

    for signature, occurrence_count in sorted(signature_occurrences.items()):
        nf = sum(x == +1 for x in signature)
        na = sum(x == -1 for x in signature)
        actual_key = (nf, na)
        canonical_key = canonical_family_key(nf, na)

        assert canonical_key in {
            (1, 1), (2, 2), (0, 3), (3, 3), (0, 6)
        }
        assert actual_key in families

        family = families[actual_key]
        unique_family_hist[canonical_key] += 1
        occurrence_family_hist[canonical_key] += occurrence_count

        cards.append(
            basis_card_for_signature(signature, family, occurrence_count)
        )

    expected_unique_hist = {
        (1, 1): 30,
        (2, 2): 90,
        (0, 3): 40,
        (3, 3): 20,
        (0, 6): 2,
    }
    expected_occurrence_hist = {
        (1, 1): 173328,
        (2, 2): 13140,
        (0, 3): 720,
        (3, 3): 420,
        (0, 6): 24,
    }
    assert dict(sorted(unique_family_hist.items())) == expected_unique_hist
    assert dict(sorted(occurrence_family_hist.items())) == expected_occurrence_hist

    gates["G3_stage1_signature_coverage"] = {
        "unique_token_signatures": len(signature_occurrences),
        "total_link_tensor_occurrences": sum(signature_occurrences.values()),
        "unique_signature_histogram": {
            str(k): v for k, v in sorted(unique_family_hist.items())
        },
        "occurrence_histogram": {
            str(k): v for k, v in sorted(occurrence_family_hist.items())
        },
        "passed": True,
    }
    print(
        "G3 PASS: all 182 token signatures mapped; "
        "187,632 local Haar tensors covered"
    )

    # ------------------------------------------------------------------
    # G4: charge-conjugation card consistency.
    # ------------------------------------------------------------------
    card_by_signature = {
        tuple(card["token_signature"]): card for card in cards
    }
    for signature, card in card_by_signature.items():
        conjugate = tuple(-x for x in signature)
        assert conjugate in card_by_signature
        conjugate_card = card_by_signature[conjugate]
        assert card["invariant_dimension"] == conjugate_card["invariant_dimension"]
        assert card["gram_inverse"] == conjugate_card["gram_inverse"]

    gates["G4_charge_conjugation_cards"] = {
        "closed_under_global_sign_flip": True,
        "representative_frequencies_need_not_match": True,
        "equal_projector_coefficients": True,
        "passed": True,
    }
    print("G4 PASS: every tensor card has a coefficient-identical charge-conjugate card")

    # ------------------------------------------------------------------
    # Write exact projector library and tensor cards.
    # ------------------------------------------------------------------
    library_payload = {
        "meta": {
            "version": VERSION,
            "group": "SU(3)",
            "formula": (
                "Integral of product matrix elements is the orthogonal "
                "projector onto invariant tensors: P=B(B^T B)^-1 B^T"
            ),
            "row_column_convention": (
                "Each U or U* contributes a left(row) and right(column) color "
                "index; evaluate the same invariant basis on both sides."
            ),
            "stage1_input": str(input_path),
            "stage1_sha256": input_sha,
        },
        "families": {
            actual_family_name(*key): serialize_family(family)
            for key, family in sorted(families.items())
        },
    }
    library_path = output_dir / "su3_local_haar_projectors.json"
    library_sha = write_json(library_path, library_payload)

    cards_payload = {
        "meta": {
            "version": VERSION,
            "event_order": EVENT_NAMES,
            "count_scope": (
                "Card counts are occurrences in the chosen charge-conjugation "
                "orbit representatives. The completed ket/bra corpus has twice "
                "the total count; individual sign-flipped card frequencies need "
                "not match because the representative convention is asymmetric."
            ),
            "token_meaning": {
                "+1": "fundamental matrix U",
                "-1": "antifundamental matrix U*",
                "0": "factor absent on this link",
            },
            "stage1_input": str(input_path),
            "stage1_sha256": input_sha,
            "projector_library": library_path.name,
            "projector_library_sha256": library_sha,
        },
        "cards": cards,
    }
    cards_path = output_dir / "y4_link_tensor_cards.json.gz"
    cards_sha = write_json_gz(cards_path, cards_payload)

    # Round-trip.
    reread_cards = read_json_gz(cards_path)
    assert len(reread_cards["cards"]) == 182
    assert sum(c["c_orbit_representative_occurrence_count"] for c in reread_cards["cards"]) == 187632
    assert json.loads(library_path.read_text())["meta"]["group"] == "SU(3)"

    gates["G5_output_round_trip"] = {
        "projector_families": len(families),
        "tensor_cards": len(cards),
        "c_orbit_representative_tensor_occurrences": sum(signature_occurrences.values()),
        "charge_conjugation_completed_tensor_occurrences": 2 * sum(signature_occurrences.values()),
        "passed": True,
    }
    print("G5 PASS: JSON/gzip round-trip and exact record counts")

    elapsed = time.time() - started

    summary = {
        "meta": {
            "version": VERSION,
            "date": "2026-06-13",
            "python": sys.version,
            "platform": platform.platform(),
            "walltime_s": elapsed,
            "hardware": "CPU",
            "a100_required": False,
        },
        "input": {
            "path": str(input_path),
            "sha256": input_sha,
            "words": len(words),
        },
        "counts": {
            "exact_projector_families_including_conjugates": len(families),
            "canonical_projector_families": 5,
            "unique_link_token_signatures": len(cards),
            "c_orbit_representative_link_tensor_occurrences": sum(signature_occurrences.values()),
            "charge_conjugation_completed_link_tensor_occurrences": 2 * sum(signature_occurrences.values()),
            "max_local_degree": 6,
            "max_invariant_dimension": 6,
        },
        "canonical_family_histograms": {
            "unique_signatures": {
                str(k): v for k, v in sorted(unique_family_hist.items())
            },
            "occurrences": {
                str(k): v for k, v in sorted(occurrence_family_hist.items())
            },
        },
        "exact_coefficients": gates["G1_exact_coefficients"],
        "gates": gates,
        "files": {
            library_path.name: {
                "sha256": library_sha,
                "families": len(families),
            },
            cards_path.name: {
                "sha256": cards_sha,
                "cards": len(cards),
            },
        },
        "scope": {
            "completed": [
                "exact invariant tensor bases",
                "exact Gram matrices and inverses",
                "balanced SU(3) Weingarten coefficients through degree 3+3",
                "unbalanced epsilon projectors for degree 3 and 6",
                "all Stage-1 link token signatures mapped to tensor cards",
            ],
            "not_completed": [
                "global plaquette trace-index contraction",
                "exact fourth-order geometry amplitudes",
                "des-Cloizeaux folded and subtraction terms",
                "H4 real-space cube-boundary test",
            ],
        },
        "interpretation": {
            "gpu_decision": (
                "The complete local Haar library consists only of rational "
                "matrices of size at most 6x6. A100 use remains unjustified."
            ),
            "next_stage": (
                "Contract the exact local projector cards through the plaquette "
                "trace-index network for nonresonant words, first reproducing "
                "the certified O(y^2) domino channel sum as a normalization gate."
            ),
        },
        "passed": True,
    }

    summary_path = output_dir / "y4_stage2_summary.json"
    summary_sha = write_json(summary_path, summary)

    print()
    print("SUMMARY")
    print(json.dumps(summary["counts"], indent=2, sort_keys=True))
    print()
    print(f"LIBRARY : {library_path}")
    print(f"CARDS   : {cards_path}")
    print(f"SUMMARY : {summary_path}")
    print(f"SUMMARY SHA256: {summary_sha}")
    print(f"WALLTIME: {elapsed:.2f} s")
    print("ALL STAGE-2 GATES PASS")
    print()
    print(
        "NEXT: exact global trace-index contraction on CPU, beginning with "
        "the O(y^2) domino regression firewall."
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Exact local SU(3) Haar projector library for Y4"
    )
    parser.add_argument("--input", type=Path, default=default_input())
    parser.add_argument("--output-dir", type=Path, default=default_output_dir())
    args, _unknown = parser.parse_known_args()
    main(args.input, args.output_dir)


SU(3) O(y^4) STAGE-2 EXACT LOCAL HAAR PROJECTOR LIBRARY
version  : 2026-06-13-stage2-v1
input    : /content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz
output   : /content/Y4_STAGE2
hardware : standard Colab CPU; A100 is not used



AssertionError: Stage-1 input not found: /content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz

In [8]:
#!/usr/bin/env python3
"""
y4_stage1_stage2_autobundle.py
================================
One-file Colab driver that rebuilds Stage 1 from the existing Stage-0 manifest
when necessary, then runs Stage 2. No separate Stage-1 upload is required.

COLAB:
    %run /content/y4_stage1_stage2_autobundle.py

Required existing input from the completed Stage-0 run:
    /content/Y4_STAGE0/y4_ordered_transition_words.json.gz

Outputs:
    /content/Y4_STAGE1/
    /content/Y4_STAGE2/
"""
from __future__ import annotations
import argparse, base64, gzip, hashlib, json, subprocess, sys, tempfile
from pathlib import Path

VERSION = "2026-06-13-autobundle-v1"
STAGE1_GZ_B64 = 'H4sIAAAAAAAC/+09a3Pbuo7f/Su4OnP3yq3s+JG6rXt9Z7M5OT3d7Wk7Tc59jMerkW05UWtLriSnycnkvy8APkRSlOO03d0vm+nUtgSCIAiAAAhKP/3L0a7Ij+ZJehSn12x7W15l6bDleV7r9jgsyugy7oeLqyhN43W4jNNsk6RRmeXhJkqTVVyU3e1ta3LoX6t1dhMtSnb+uz9sszze5nERp2VUJlnaEb2wKF2yJC3jfBMvk6iMO3Ea55e3bJFttsk6ztkqy1l5Fbfe+7f/ddxmWRp3VuvdDYtXq3hRJtdx59dok6zLLE2ilG3z7DKPNt1W6827D79ftDr417q4itk5jq7TY1m+jPN42fkKX5gc17jVYvB3tMiAlLQ8+udxeH5x8vqsdwR8ES3CMo/SIkHqQ2xcdD8VWdq9/KNF+ItFnmxLFq2LjMHtzwXLdmWRLGN2mq2jOfualFes00nS7a6kUXc6AAA/OsskB3rf/36hCO4QNZKIPhKxd1YkIbVWMU5AWCTp5Touwzz+BCzjdDt6gLHnAC7G93md7MMshKXYbTZRfstRtv7+68kFu/j1zTkjQPbh4/u/nZ2zI3b6/jcY3Nm5GN6ev9YvMN/xdZzf2jPGaMaQc/w2cCSH+0D2p90lCRVAzpOSZSuWlAWjsbfKPInWSXnLiuQyZVGBHxuY4iIAqUoKMW00/0/YMkaxy4oYWkeLK7ZO0s8MBKIAmkCyljuQZkK7vkWhzYRoJzkId8H8bfClHRCiPP6yS0DcgVzegFB9TQqQEz4ZLFqB0KNksxVM55rN8whwFnGOI5FYNtk1IJFj6ETLTQIjmK9jNgcpslQqQdL5XX2chApnFkQyWq9Bs4DcNajasiJK10AmVXMblVeFoFNRVrB+MAiGHCtya1fGgtcsXoN45cmCkQ4ncP0s/MQFH8eJnVwCLpixa6AMJ+xsNBmdycEuUKkIsrzK4xiuFNkaIZkm9ICz1wG0E78/6pyNwk/toxFHsFpHlwXbZMt43Sm20YIjSKN0gYQA5KQ/AsxRKWcH0Bad03WW/BFHYFBW2XoJPEFGFCZJ19Fit9sYTBIYe3XaTzsgnmCIriLg69M+CexpJ1suxaUi/ByXT4oQ5lvQDWP9A0WF2LmOb3CmwdR8BmKIrx1NM5nSzLq2/fwedO3d+wupbQ8qW+sNsDaDrrGRmEt2uo7nxeKq8xpGBNK7yMDQJotEqAyQEYMNW8Y3TKMK+PVrFOUsAvqTcgd8DQQ7j4rdXNFu4sp06RfW/WucXF6BhWp9NEUbRZFtdusygQ4WYIORXzCDSoAXwOZkieuHlF4TOmBpVmr0vWrRUI7AkpTWWGBRuBWqx6Llkgw+UKh6AOpav558/PnvJx8rFrd+ByIiBlYxXUZgprjNP/3wO8t3IOqbuMtwkTjp93oMrA4SsytiXPqEGUJbB4hP3789+Xexan3cpbjcsfk6W3wWekg2kfXEevUnQF4tWocv4bjgt1Z5tmFhuNqVuzwOQ4bmI8elCYgjrhetlryWX26jvIjl78s/kq38DkJ9tU7m8mcCVJZZti7kBVoZxPcNTKP8vgUTBGv7Rv4ublUT5Bcnb5Gt0abQtIibp9kO9TAA7V1FMMfLBKw8Aa/UDArQX8QFcXuXLogyeXud78IFmHnRF8oYDETe/YC00o3ydgs22+ofTKz8enG7jV2QPwNlAXsDEBHY5IC9BbUN2DkYnxhsUsAudqDtrdbfzj6ev3n/jk2YN+gNRp3eqNMfdvhMdq77MFMf1hG0KUE5J7zRFExRwOr/zVpvccVqhpq13uBaVYOYtc7I7bqAxXDd0M0M/Dm44/v9gPXgXztgPnz2q6/4q91unfXCEQCCtWU/gbWevDgaBuwKx8xG8Jtugai3YALlJIbkFvlt1vkrMX5M4r0gJZrQFd/7NufMaxOqZMWxdcG8FmXht3kP+JfHoAApv00XQd1A33m33cXXJZB1xDzVq4e/DulZdkz4mjum23QRbEtZDXcDQwV7E1XjDcGDunjEoEUHiNZiN/c8Q/A8bZ4DvSa7vXYz6Q0T0zf7d/MRBZtougYj6+/GTpEL2LX7BlHtujHWe/Z3096MPWXX8BGw3bTPf/Tpx4D/GMzaQMi/KWPgb6KbAlyCyTuwvW2icA56Dlb91t+OmdJFjQL+P6qepLrb7QpKbmBCt9PxcEa/ooDN6cIQgLbTY371JoJrxIabgJ1Noxln383cuDyfGVz11WT4/g2N74YGdgMjClgEKvm0T96oAoo4VMTBIoKbO+DmHG7O4eYKX8eCq3U614EO5mqIFrzOWrSeGk+NeaUW1ay0sa+fWOfH/QG25tiV5WDkf3B/XBUokAB+bPwkHzMy1cQLYAAf/zZgX0AoktyQhC0Icr8NrqT/RX3Da/hrAEp3xAaig8UgTHeN2MEt+MhRDp+cDiiceUU+GsVBpnd/i24MR3c06qJD0UjfFujh5Dwhkrbicyiu4+eXh6RlBe4S+Cow4xjVRGtrDFwF6beufUAXDUMGcbB24WKl+Ya3HfS/5Tzv1nHDWMBijmkF553M4M6UMx7A2jNpO7fsr+idSS2BVt1ou43TpQ/AHQKmKVKr0pc9DRC4o4ClNceR+gDXPohlEXif38+2XtD/kWyTXJgdwgXBY40Th/H5e9hWsQlinuxznI5RS/axDCLuGF1XBEav3u/0ybLqNDhkuI2D4Y0mEwwX4zUEEg0z9yDpMtGjT1S4QC+18FPqecxXylRhrwamzAC4nuAyjHX3tpo9cdG/I4evPWb9ez5GytOp0T/tB2iGeKcwg8gOfkF1rCYvvSkf6KytYLGXJDdFETvkNHch+tgYjkrVZoBgYm7FpFpwgpYpwIJrMDH6UIC8I6ANIPW55dennCuzhyYKDaqK0ijP4hNFheHtoHTV3AzdUa+7G1IL+eqlpXpoaREpHWQIBpZIRZdHkrx3DOZAiG7AxIOlBxkQw/4ck3clUkCoXdWPgf5jqP84DjCpJTo4YX/EeSYkZBOD38pzMSugE8ihNARGxWW2W1ypRafLzmQqCcN9nmeJl1zjzMQUUjWgbMuwy96n61sxWLAJFJOlhLSefQMSOTrCX0ZJGi+7Bi+Faq/jVExSG3V1pN+L1mu/VGrfI83nGkEd81YtQ7vIvxFmRBd+a/5nqAd3ldsltK5SQ00F71tKEzEHhVHVMr4JKrWMYbWGaLSM5UAsJXwMTVr47QtYMUJdT6/A9GcoSg8pKY9IULFQP02z2Kusoqa8TTouMNV1W/gtk8oLGrRrQIiIE31btzJqNO4xyD+gXeM/iQUKZ8CGDQ3wT3Q6APrEVzSbnOag7WyGXHkMQieotHhTCV+3fW7jx6VZ6NNETu5lXPq+SA8o8922dQV16XN8S4o0JJ7DLxIRjs+xdIMfAnbMFwByAsj1J882S68xWxxyjvFVL84LHiat45W1xuhmdMYjlhxzj2OVoGm0twBOVrkR3bhyfRphaqsbMgHCww3uAhC9dRlDkDmAzBGEU2vr0NSPeMjLg7iIh7w8lIt4yIsBXZtPcgQr8mau8xpQCIZywxxyQ8G5yC8VGotUyCZYiLsO+n1anvgMgDVHTxDiuZYdvFfLl1wEyWPExugvzhR/hD3bmrZMkKUxKkkXyZISThM7zGyThK4pruxVahXfwEyk0TrUt5MmEM2iMnM1BlF9xg3R075qJ2yUu/0Tzo8ptZ/Bz4ouwip/ENKeibOQ/qxm7Ax9UIsKzVYEfd/+AU6j2vjxRRNcMccUMRtTZLFfTAqCiUg7y1CreV8cy9TDi55IQ9BunATg/2/5mkfTI5vIVFW1gSQRcDtd64NflkBK5Di9St441bhaIE2wXjzlBIHJ5AgCsRjhTKPFEubjjkJZRaVATxe4TIR4WUst3As8l+tsDtMrd7YO0+w7kRXV/WRK94WbjDbnxk3LrMpn1Q0F+Y40+yHOZYTpe6kySm80xQHvM6Q2y/gSY7aJEDW6TsQk6XWUJ+CYY+qB7iuNI3ahQUI2ji0RBVDTSgh2BlzIAmpV6ZgIBmAuYPEtRbxTc5IqaBkp6C06e1qo8amoo8KhgOrcgCu+dTUQuDSHhjuTk2bPvaIDdBs9WbppGmihwHe1hdgzN+pXcUT7uN6Y/RKBcQjqDVZRAqRcEtUAhluCZNXaD8CKqZJNBO2ORpyJAMe/OCEkdwlK/jAh71uaVTbFDCfV8LBwZsX6Xfe9iJ0Vk4UzYeGcHBAC69Fv25QLpzYo+ajfDcz+NXFZRGmWJgtaELBvxJOkvouEgPka8wS72xVlhsGY+jZmWudrF3G9N2mjFb9auJxWRK45umhog7JsIHrSTV6XBRrI6WtxRa22JcXWubBfuP5opkts4JcVibwHZcNg1A/C8N175+0qXGoQOrFqWKNRzmel3dVSSLtfHeh1vYsJN/9GHgt20tY2UQTlm6j4LPTB59AgyIinzf7yF/aJsHwKdETS9REoK5xisE6MvcPRVToLIzUCUNL8s5G0H6KFqfOePsECOSwLss3SBtfNqaG40EL/aTUz+Aegxm8LVuMLQGq/KjjdUq1MZKYVR56ALxR9DVWFSiiqgzwUXkzzuBqImqCQV0aEVGhCDS7yXWytEw0ybahwPfwDmgCd3Pb2B/0RrGQYDzyBjz7/GMzajxpNUeY+3G0/akS0ahkt9mkqjUpnvjY/5gAbdNlE4DAv0rAh3UK6a2tx8xqM81PJiUcpKhgu+UQo1BDM0ndNrj3Lo0BJtnwMC7a+usg2jnWnaitM0z4VQitgWzBSfjRhGip9oQlLcEhxcFOD/3dNfkKoViBoA7E8rkr7PIYafN8F71rMBfzABc+dMxQJgKJP2xmxUz2AKuCQmrk3Vlxl7FXbmc4z1zKKPBeYXLd1not10xJoIVTWpOntlC7ZCArdEFpotfa6OrpQ7FFXDYtQRxcCt6ZqbXUtNVo61DeQ6c0fvsv7OoZZLvNbrfaOXcXrLfgw/yM7vLzeLCxix443XDy8ZqBmvm4CzQnQSglcl+e1y2YTHeReUE6ivJ9uSZOKsRtC6iplV6LFpcJNf19ySUsYzbNsDQyB/0XKaJcmX9C/kSqHJMqMENfZq6ggw2usS3ixANUOM9wq0O9QoolSGFF6GVOqkvfRtlJxnyqghG9SumFlGZBkIIeYJjCsf61d/TRz5Ii1EdR8hXidGGJlYK9fb8SvM0N1ImZU9h+YkGIei6to8GwUrhJMAWEhEVX8cNEoc97bFWaheaVgl8OLbAZVB2OrbgYrtO/lc6+NtXWwni3XscnwxdWOZyOwyNBfR5v5MhoLyG4eR0u/jx7uAHPPc8+zBnrV3W2xftMnNEZG7ap7Fd8sk8sYvFOZUfua4zCxoEobVMCy+acx/hcvSnOE3PeiAqzlbrMtfAAKKHWYlpNBQOIZwnJTTMidwDR49jUEAzsh0Wt3QeazZex7u3LVeSEqqIgxnJL5bQkLtvLEqpnReYq3944lvPzje4ZTxNuIO/MT3wu8gHljr/2NY6OZx0rS7mv47xeUHhShNNrEE3Q6kUwsOoCGE+/r3AvIRsMaUazj63g9eQn3sFJ00nMKjBALGnqNbbbISkahEDn4RKzhXBpbtJPUbomXXl4CkTRU8FsmcrAu4gQZxN51BoLLASQZuB6GdJgh1EuPrmPflWBvTqcXKiqFG/4NT8/e8H2WS5mySuNLBda50UAMjmH2Aquo40tJ5CaCS1S3GRoSpUoLNd5hSKSKC9BMY49YFY3/+Xo+GBt2N5+xMhFEDR0wIVNUhBhmn+mn8OG3OY7Lm3gQ2fR7Ip0vLvLSFFFXThWHnT47+8fJ6QU7/fXk3buzt+yI/Xz27v1vb96dXLz/yH47effml7PzC6/9EO6VB3a1wAw/G7M7Ub9775kg/KAPI5CKSzaUSIITVMUAE8q7ivLlV9yfHjsKzF/VisqNxoJTKl0mCVFFnQFbefKkDacZ8awyqg0xKJfzdxn3QlAgXHI1Naog2xogABk6ZUNR2SrfPQToqUe/vZm9706XabfweDAQod6lto8O9iIQCkr75SKWB/+uNxaHU5yn0LCSj0Xp4gosWtdglKrEo3LnNt+pbIDgdc/7IPoK4kUDxED1YhQWiK1qjYjjhtsC/0vX7Qp5v2fIAzlLuK3uizLujkBzJ/dwOer7vY2e6o2wUIx3qCbh4mvGouWnaIFnebbSq4O5w1ITqiPT6pzEjg2dyEjmeCLlbHL8SmCiEg5QB3F2SR5VKvCgBm60DI+g5wHWmz87GvIJlZGxlrXUYv4+pthlvuSYqK9uvqjdBDdPXTB5H7CR3nQe5cODWvdcrUcHNeWzWjU1Jqk+7G8cdx9LF/YNrX+MhQ1N1PefV3e1+pRt5TuPLLf6i/sWedE57vhh+aCZjpKCSUUdUqX05U6VjkhRxZpTvdr2W/F1GvG1zC0PYWQs6QtxHvZtWfStqRmEgwdaDKzpGoa9B1oMSYv1FsOHW5gz/mAfo6qP+5Zz/0YxyMGhfo0Dg9oI+7URjGoUPjPkkBaQqfe6F57/PgzNlSHElcEza6+88msWKvMV2hoG+O8+jylfes1FBeKla5QWG1KmlO41Ap3MEzzxxnU+aU23yMulkae8192H1z324eT8fCyWQJBdPGHKQFDjtKDTe+w0KpJNgju3WEC3xPxLps6BqgEKoZYnyfiZk6adm0/83Nb+DR55mrF+V5SDwIKg1TQUalOH36n85OqOzNs6b3Ky4qXzJsdZS2ZpjZsSZs79qGYoCI9CBWYRsUlugLymuzp+jMDt+zQjIUYnoZqBaqerlmy8ItbrJQii5sasOMAsNE9aP6ZFJbIHNjLyz9lisQN7iod3v6VIwrkD9RjqRe61lk83yifc2VuzwuJbKy9wKmUBJ9chc6+Opjrg8dSkry2UJNqmztzx6E5fwWS5jTxbHtrNvNm9Vu5h6+HTCfnldiNt5/AndX5fb7fZFSWbx2yxzvD4K7h9cS72lcWBfqZVUHUNx4BiVhm86r2OXYu3I6ClmNeFoKJa6ZOsF9oTinNuOim6rxUqDMBBQI7xDshncDPQ5LlupwTPBYaK5gdLtbgnog7wf0/R1sGFW3raMqglGM3Ub1XDpVGpVXK1rfnZv6ZIO1hZwMo+qnuVSXPfU4rpvu0w6lq3btutAAwjXsEYss7HqdRfFzzTDAhpMFVAqlDsTOyY6CzPF+sIiwQpcpQUcr4HezGovsPDcCl4bZa5/Tit2QNWfI3kgzboCRhERo7PyBAeDCu2eEot6rpMgiRo2rTVO0O1rA+gGd4kWUwaJrdW3p3Qiz8rZVr+eXbfOb3TpnbcGyxFXkWxL8yWy5A/KmJiMZrv5FvXnllEiOqzh2mvbwM4vSOj5KBJv5xANR9Q7sLfOQvRPck+8GTl16AJUrKUdjsto7X0Zg3taFwm/2RRiiXSDQj4EqWKfFXrSoibGlbTilvT1S83/JMncv7q9+/rhwHQnCbpLm7tl/qGLddGqW+C39/LgxUJjf093NIcoNPVr4liyq101c2ejW9zaGlK3nPVcv+uuUMR+SOAloTJpW/i+TThYg1sxCdAkPvv1SDdUUFtqLQXiJ0e1JcehtS7dAUpjg5dJ1AcfdmBS72/ptDGrO3ZF6c91aeqobBi5q6zciJDehprmNwtUvFgLfPIkRVxTR+mclZntRWEaUjsUqOm1mZAZiNw1B9JRLWqGR6joS9RIXFUDzmWGDzmM9GeXWDQSG2n9YqiBnNewTuqih5o46wscrSp29qmKHUKQ+OnakQXWjHSrM5CNeUmF5tM7th5UG0pYw8CmzbVXc7ch8SaY+Tp0jXzD4TFWCddN1x72tS5/ZilwGzdrlHaGJs30ulusY/KJvV9iLYDDiDsUW8HvsbCdedA91YU1kfZaB72DbMeZSE51tVHDNMZmRko1Y3gENobsBuBnYG+uhMcIAEO9I7FFMuYfFxTHOCOKBFXGHDttVaO8ONBX/swP/u7fOVv8pPtRhg4WIWfA1erEM/aKte67wQ52Pn2LN+lqqaUV+ptmpz1+7ZR1GzFTOPalNcSt6YJrioUwwKmOK6to/0e/EGAaCmZAfOUoHQgpRc1OB2qkn4LTGWkpOBVoqwNnycznWcJHhnL8cRWBcnzXEEDzip/Nd6b3LLb81SW1kbktiw4meDCcgjcMApFXssbV7mveguVCAPwHOyH1UrdtpqamVpuEkShsEgCWQ3MwL/eQM1Y7VxFPQ0A7Sz5tRopSTGL3m2D39TIWf3uMO0N7RvLqF3m24GDD1Lqn9D3asyGGXa1bwheJIa6LbdPwFi6TeXr5qWaiCdqr1J1ZkyrfrKlWv31rTu5RijtNDWWP2fUuY54+HxS6BDruoLv0+fHD/2xUvYdAvadsvW/qoHfYVoeZcdcyxo/If8n9gzWFjxpZi5ufB+6HsF4U1m/M+ML4OSuwjYO7o/uqtov+Mk8BwrOnMldbbsEG4gHXk/uXAxtwhivo20RLwFlVaDIOrJwcdztr+4Lr+4FrNa74mpiHhXSVkGlT7ixVPECgsYJL2RmN2NrPe/cTOvaYanPTZNutVvV/xgqosIG/DifvY1IdLl2EuXRMFB13F/BL6qurt95/UxU1gFd210BzL7E4ly1baeeiFXbw5+w4fD58EUdytiVx8eEvhgZ9WzuvX0CHD7TAd37/NCvUT3n3i2fsMHALBChB+rarqnQSqtERJRH6vuo4zoLtOKNZszjOlMOKfqQRA/Ew9+5Xsvsf7Iu49wmWrFV9exitHH6SLBXNXAx3NXAyZ8B7WrsReCaKmER7cuHV8ZUO24wx0evB6JM5q42X2gqdAnu/FWzHCvPbXxOO/z7K+Y0P/dCe+QTdISt8lqa0TCUw1FPAgL9fGSqyL7yE1CU58d9o0b1gYIV0IXe4Jn9qKv6FueEPT9+YSh0Qx0LIHx+/LKJAqOkZcJGz16+MDVxGOoPYZcU09nyWrGWm2lKrWp3Hn26bh+Gx5yv24fHwW1o7biqHyd1sR4Pk7quN9BsOrENd75N04ammtVGj7oj3ifAH9hV0nNVQI8MnWtmH1/5+d2A3e1lNcJqAIb2xTdbrrX1ZG/tmWxDrNMcs+eDXmBcHwXsGVwfHGuX+/TQbATvPx8OBy+0WwN6TNgAbw37xzquIfUxgjvHsg+jkJKebuQ4v6pTXT00C7Vrz/AMRbbzlVaFemP+0QHXnD0G0zR4WYNvyuFO2MuXpmEwX19SeU22VWhiDRZvmjl6z7/7PO3N7gP46M/u269geBP4Ppjd4/HW+kliUex56BxU/vT/walxbTL2PnzhoOz9/ijpkMz6weYDJlpYDzwTg49L5w9rF2e14ep1LJ8DNBLvFpLpYlXvyka6KQGzx0/QiKcZ0M4WYI7m4BZfyc3YtnR8/47nyeQzBrtmPSy9pWKiHWQST4o/5N09nlafWjRian6fj2dEGc2k7HnJjzjgw9/o04jB8eIfz6oL5oeDzEOHBpMCXSU3cRnV1E+esIIb4nyVFbIKR5LOE3m86lo7W+QG5seVCFr+tgN2+Rgb+YYaAPboJTWv6IEsk/7IczyFhD81w2zVP/LxTTWf2pPRkXpXTb2xUXReU7IwKcI0K0P14pLa0yjubSeZVE7P8gh1ahsCZk+Rr0veD5gefMIrVtADI2QZaPUmpl1+nVxjrT8dowT9s16RxOvUpWMstMdzj1m6zeN6uZExcKUZLuE01OYHjB4jZTF6MFIL8QhaES2LqvpqjXpFuzrAMWBMvJQvjPEa55hI5cEPD91belrmJ/YRz+510GsiK8nXSMtU5QhDINYBPUNPLdY1NjL417YP79U7Vef8qiN99QN/Vp/TaujNzdSB2sIQaDocHpbxTQkL71TJDMczPDZdiWfy7RjUdYh9216EqWDemGhxTFLT/Xr8PDx+zIGOZ2IRFJW4eAgctyXxQDLL1ezzwxxc6kQFOT+NIw9yiEyXeRa3SnXpy4E1/m/SiiXfCNTecmObQ/5CQjTQt0VXoLNBxGuDAEh+7covfruW1+QHZ7HX0w+/291FsOyHYtlfup+1532FWAsZE+IsCYa5NVMuRRZXUP4eXqAeXJlklk9KVJUZdVMjZrpGjuhAbSYdnKn6nmzVnscrHZpzcqB4bBZqH4rHJqTspFSU3j4iMaX7Gjy6fWSG4sdkKX5kpuJbsxXfk7H43qyFqS8qZKzpzCOisu+NzH58dPZjIrQ9D6uiijF666o3NtIQzuLHKgexf1evEatVB7kfX53Ve/GaFZKNqJsL51yIm6FdHZgrShnnW3wpgbAqdw0VASJKKHby9Yq44I1eqBNY9rtPeZxcd7ybXW6+T6A95i+EZTVETyucx7Rn6I0dZZ1evxcMhs+VbVdJcX4CCEsf0XGLIJDHF1I4X97p2KTz/OfB8YsXaFk6UsXxXUfB8+OX3Gi8Eu+CXYDPcclfQMBzRxm9eg8dbu2BbGYP9nRfbncgcYtEODeOUeKrHc2EBL0OU5TJ4ks16NwTPe5APHG6KTcBMF3XiLGLFBxYtedWvRJ0m2ewYNxu8dEI2W69xJdXImvB4zk6RwH45e2bdxfiOSE7fNsrf1bILcz0p11RJqskXnYbeaBLJPnIwAP61K7jE0DqzoYRS3TxwT4AUfk6er4gqKJX9JXvg1b9JEnRgEeLaQNR3cZdaguLEaXU0egRYjM5OjdwPng+pCYW3htcTnguSXv8iJrxzjzCk1HGe0dX6o2ooJx6XotRObUlFZ7fD8SzNgb4MaQ3W8HHED9G9AuwoYaV+C5j58tu9TeyahLQPiQYkYmiWipBzyAFEsx4Yo/1oJ7ffjv5+E/jqTXac6dE+6n0aGfND9Vqt5v6wDfivj37x5uLxm7u/j8J63z+58MJ9mlzUl3UI9x/w5StPPk8Jnw0kmFJ7GcofTz7j7PTi3N6hpJuKWy4v7//+J9v33CEhi2wAYVEEkJdmJvgzn89AROiAYNK1Do/efv24s1vZ4hThJHj7mB1zwrzWU8App5W9frk4uycQn2viU3V7L07+8eFfNaRbWZoNTJsTUbLg77SeD9nfFmEoAXfwJSRLaI1AxaKrkqDt5IVC0O0nmGI6RMvDHG1CUNPvKIQXxSMz2mRLw3unuSXOxSND3TH157Ty1++DgRNPP11izJFRs+iqLwnSpnBQpOTpfznsXwZsqeVuvDeu9ESQjrRre91Ojw2D8iQTvhzysT7iybWG2j3ouH5ls4yyffj0l+vKtJh+WURsHCXfk6zr3ioTPRAHyFdxJ4K9QSCJPXxd5fICqh9t0Lbbv03TEB7asuBAAA='
STAGE2_GZ_B64 = 'H4sIAAAAAAAC/9U97W7bOLb//RRcLRawWtnxR+qZ9UwGyHQznS6mH2jSnV0YHkG26ViNLXkkuUnazf99l/tW90nuOYekRFKUnXYSYG+BtpZ0SB4enm8eSn/+09Euz45mcXLEk49se1us0mTY8jyvdXsc5kV0yQchv4nmRbiKoixcx7Msym6729vWyf4/rdYZNmPrdB6t2fn79tBnP0MXnW2WfuDzIs3YPN1s4zXP2BIuihVnb9q3vx37LE14Z7ne3TAAvcyiTbfVevf+NTv96eLsHTu/OH1xxvqtjv2nxeDPX7Jdwo7maVLwpDg6NIXWz6fv/vbr6buzsrfW+5yziEGrZBFlC/Y8XUcz9vztewYdF/GGd9kFIHra7/VYnLMkLdgu5wvA8OXrt+8vWhUmJRL/Og4J5z6iM19FScLX4YIn6SZOIiBDuImSeMnzovshT5Pu5adW6837i7Kzht4GR/luGBJxxbRKsubUT1MrwGEdJ1ch3M5h7DlMMi8Hbm4jyZjvNhukHY3Q+vXn0wt28fPLc7koz9+8ArTPzutro6j7Eyw0/8izW8kWiArbpBsYjy2jTby+Zfku+xh/jJNLdo5jsn4ArAGkzudZvC2AZ5K8yHbzIm8hx9C6El+xkgAtmsf//ud/2OI9Y+/a7312At1uws9RMLtj/74Iox9Y+8Vvnzv9Ox9uwr3vL8LZv4NW63rFM84+A8QdLm+UwADbdTyPCxYnH6MsjgBRQTo2i3ICWbRehMAjJ9hJFEBHPwA3IJMs449czCrmAJjxscCs3Q/6PmMLvi4i0Uug9b4AJkvyOE2Axwl6EAwAesuzza6ICnjQOdhyIFoOgx605Ns8XsPNwyMNg+GXjjQSLUc00iLdzda8c48BnwkKyTWN1nnKNtEWiLRes/63A4DMiziBlSUe6PQFoxTpFU9YHl+C3OwyoGmRCgZoySUhboYBSZlsV7d5jEwGDAeDp9mCZ3IFrniBeOU8w1mGff1ioF8M9YvjgIHmQFF/9fbNu4vT1xfs/Pmbt2c2u8PUkF+Je1HF7QpEtWTWX948P/1FsKxAO++yl0CclAt9csuJy4sMgaFZa7uOft/xouAM7/FOnCz4DUt4cZ1mVyyaZ2meE4GIG6uO5NjAhwlQQejW1jWPL1dFDuQABsw7z9dp/IlHoGuX6XrBF4BStslhjqewErBsxWrDi3iOwiCwf0sWgv2E2AFRjmJQFpegwivYLlmP1jJLNywMlztcqjBk8WabZrDWCaBGzJW3WupedrmNspyr68tP8Vb9XkX5CjS2uowBvyJN17m6QZpI/gZCFWBKNuo6vy3BUHULlObpes0J91zh9DwF3c6zACiyjHbrYhEDSxHwUk6zBFXzFo+3UYHIqYdv4VI8KG63qMHk/b9BfwGsMc8ikI+A/QLcHbBzDquazOH6Yrdd81brH2fvzl++eQ2axBv0BqNOb9TpDztC83Y+9r0WPhq2zv5x9voifH366uwcrifE0B5wtBeInxpf128N6reG9VvH6hbwO/ycgrSi6J0ryYNxCecJrD5KiPOfaetVVGTxDQDjhCf0j6LfFPpsAbkVycM4AV5t+6zzA5FxTOPPgZ/jRYTyo2aKfxCg7X29gfX8wOirO79ewNBHzCu78vDqvt0FNmYbwArQjirUQjCLF4OvRbC5yy/tcUr/or+FvIuqsqLxuBwzXtLjLr+BNcvbfvUE/2QceCAhiJZ2XXU06dmrm+4KWN5wEWf2EsNQ5nJ6vmNYOUTDwg88v2VB1Zd0ADqp9WfWebg/0JtwcTeCz6P1JQeBeeBBiI4xrG0RF7ehGKqdjFHCiJRCyMY6ASpJKeWtDeDtmJ2csA9+wMD9QBb4gOufRcklbyf+tGyEj2L9keQcgYrAAOQV3Lict8XluNRmk/IHjMj+zSqBr2GbgFCveSK7EKNEOeogePQD65E1A4+gjUBZeu0j+gmhB1eIoGopmu4ux6xUOS4a3Ihp31RNJ/F0yp7WyevDg8P0UA/BpOiPK7bdxh/BDgPS/KZot0Wck1Wg0C5giY8yAMhPsukE7kzZn05YDxbpNYRBftmVJIzoUUYeCBEwLwdTs1uDOyGw96o20Cn2GNAvajoFbKqLoARplY0+SQi8Kx7Z3SF1b0C0PlXUVI+0frRlqhNGSj4+hzWFluYj0v0g5HGy48YDhRw0tJCTPX6qd6TgDaZwAUw+TFkHhnhSzf/D1NnCEp4BNEn8GqQkhhJKGGKSjKc6YWAcS6w2oC3j7fq2HSleBqdT/axJkGSKiCRlRv+irESggElY8GLmN6gGA18Ij6BZPJ1cTWEyM/gPqYGoXlXzFN355kwtWhAMjl9BNUoS4eo71QsAXH2pboGbpU1B8dholNJI0JPqQlcWbh1BXFNTN2I6m6DUXxEIa0l2QWzAHh72/riKwJ4CthE6Yr+GQKOttAM+MSWhFKdKnCfYOeoATTVUikE8NdWCuGdpBXHTpRTEk306YePWCTA/IiEZgErY67ItsXJqA4ncXsm35V5O8H5i3yzyJQ88PZGxtZocMcYJ25hTmWU8ujK0BQJKiRBCXhpcTIhVukHeNTREtfhSP9i6Bbsom5KisA0gsjNC+f6+fmQPASPQA/2IyWBAFebwLLlsg2QrsSO84bbhxsB1+6ab7DYQN4FPSzJw09W8XByxz/g6hxjX+6yB3h19NgDvPFO7CATyUsFoxKtCFQCaTk2/amKgfw9N8fAe59sqOyPC/Vm0jkAlLtj7o/dPalmq/KG9UW18MwrsdrtKh0u+CLVMUns71lEXQUB1LcgMUQKKag9NEHLO1i/1Zxywj9F6x5G4XK4zB4BKjqDxhEBQ3GN92QpEsg3PFRNiYiQ18cstBAP2+z0Q9jxvy6IlBPbs9y5mPWqjbie/k4/psHxbv5QKDZFwfjtf89xFr9K+5Zwj8Sc/RcD6FrHmmMsg26NIlxcRJk6soY1gDzucENy0wWpUN7B7Q619wOwqti3vXK/iNScTTB1/sDqVN5F9Msu/w762Sv2quBKHlIRSzB6KZHH7qoqDMGkjxtGXFU00CHO7zBt1jUUXJLlSPo2uW5HAmo+fx5ebCKmot9eImHwMBciJk/3pWWUvQE2MmZkQEePppqaIds0DCgKt4fZHTMc4WbpEKsC+TGsFGHSj7ZYni3bp+7xmT564eFGNQ5Fj1Q1tzcg+MDwTxJKzB5ysSJHsgGZOjAeVORK9iOS6brc/G+h7BBBSKtQDJgjMp8XtlsNtj/LX4TaKUWN7FtBylyzCIg1BW8b0W5s6NCa+2fp2oxRg1hF2vs3SxW5ehJlI5bexD3DRwHeSHU6wQ7gz9bWh70xvOGBbS6dpK2hEmlIcKjp4QggAlaXX/nx1F8BffSQvCRELeH5l3FToWQ8W/DLjSDUMZ/QHgtSKpDp/0Lz1EUvzE5a5fqQkqhx9WnbnACM2C6r7yBRwm3jDvKv4BpddOiAtQdhHyO/IrQxpTMng7pLS5IpdzXLS+WOkfuRuyhAjQjK0M/n/vNJ9euSDxP4MEj8DiDuffc+GjgioElSpJDE9FLEf2MzHdIi8mKuLGV3oXNjp41haF38BtiFP7Gm/ZSKudLXgRol8xYUuDS79TSB5wRdt2VJv5JO3OekFbDh1au4qoBuh0ppOdd3kBuwHbKQA76Gh0vl8t43BFKFgYM6eRIpiT0IXERwKknilxN1LkkV7EOdqvg1yLQnTINzVhUvKh40SLhfusFz3HXJsBlqfaxGSpbh7QR2iGY8SJF+nBY0GHADrNpg2wIT5ls9jjqDGejnANcUuB24TkuCOBuJHX/0YGArdVOrTP6zDNNkBy5VBOEWmuMyut6s4RbjflhMe2F65imLAQb0Ax3GX0FYoqK+y95yl4ASCQBBdUSBTVlzD3wziPJ6Xru0uiZVxz3khvaeRdAmk967vMMSUyah8L/BSZhiRab4XSR2soOaQLuMsx66EB62UQC9AbYR9ai5IzsFDXdiwJZodQpP609NWgKnyW8TDQHZkhruoSTFqoDCzp0su3G1cKIpB2sIRVTfH7J4rFUg1BaqQ51riC7fTgpap61HHo7usBtERLI3Gk7bsDKKQKgiJQH8+qYBKwrihZQbRN6c8ely9PpJej7EFuF8sZPyzTrOQ2MHl/AuXTfLea2C8jG95VJwoJi77Cz+KYhpzK6FpucvrgOlpAR0Z07cvG5hbcMbOwm69DqWyKEw0aqnbevbcngfmdjOMFetPPuCTlivZlZlhoz4dKwt8MCdczfG+meFaC0kXMxlFiWKDVMRCz1oyUsa9fr6QwRaKkamjKmqIqopGDCr1BMNGa9KAom/QShOrm2pe+W5Wyz5ODHRxYWSqnUhG3U9r5IzLZw7qxWbeHAb1wWPDKSgkfTsEF7eVGrTwN5LKRjdE28bkpZJvRRkpy2ihA3YcsGdyCW1vzbFX56CRuW+pBnHwkHqk7Xg/TFR6H59v5Pb5yoC24j3NEwpcjFjFhI5VNOwWJgCq3fdGZqQB1ZJ/mZ+m41qHtUYEeOvOHi9PFI2Fe5w9mmoofBEVlwub7vL6iCAWtLTuj+Ik6tj5T9RdA41mb9H/LwoIRo0BQeMSNcQFzzSICMS4YobK46yFChNaKdpLwx8zX+hE8nLqNtJ2tashpNNtdY5R7L6EvaFuDoyAOtaapRLQ0OB79KQAUD38f5P1eFluIgitxdHJESl/TIC4am5BWQGv80fJgAhmFI6WkIsx+ZSBrpWk/0l7U+qO4T6bznOyBJ0pOpso8RGKMomMJ6X4TO3QQBtKVKYswRNIImNjfr1u99j3J8Bz37PXWvGJ1tQ0D2pgwSDTiTZDgcBVTOregBOCKm0rGCMBc+LK1WkGhOyVjspknCy1fV2YuPU8WY6r59g3PCcEJvtzqVUjsYej7x5YbmaytDwV0eKJyFDJDCvODQeq8qt2hota1emhNFgtI1bFS/Wl0TuwNGE9ZlXk0E3WtB6uSjDTWE2bsdKZzYjNZGzrjOZERNfUUtrEmimKYvBe/oH0O8uyNGvj3Mt9KiXuIYr7rRjKEMmW3FsImwUxUHHabpPsBSOJVQ6hIDVYGY35XdZHEHHNl2bIVPqVdWUSoI2xUKadDoNHo4pHYTBfcy0zLKi+/2CYl63NvT7erHE8mLVGA8M2mDsFRmCIFJlEGAACMPyYzPA34Q4/D051H1p17qA6dUNXu9aSHA9tKtITmTbOwWY/RdIbjLoNNG/MvFNj2C+GrKbx0JZUHV+gkwsdPJ6A5xywNvwxDCU4H2mCRx5kaia84vdIz5iV3GbFg5mCc+Vr1OLDyu6qgZNoww+PbBd8NPu6Ld0TQDKGsMpheRJEsEh5OWZmuXrQoKfQ288y1DghbTILVFtW0kqYszSP1YbyJFZbd3aYpgYUpSpoMZ72pd9Q2sWv7KjTn2qegji3VYWTD+cVlAeb0LiYzohpmL/K/ousaGyl1bTEiE4hYy9Vu62s/r1yVmZbK2tl9qBR1hkkuwPlpmB5Ytx0bFDce6O6BKaDTOVkCBqDCPp/bxMUxxLcXQpH9NeOloCJCJh+PZtOGxvqgRoN44R00ODOSiLSJV/v99l0EbJkChpadygZU2ei1n0X/bF2sMy1pGVUvxuhcRlpBfV10X26sgvnfliZ5ajSVi5iVZmrA1kLc50aXeMHXS45CSP3tn8SX6HalDM/qXSOsR/i8vGnVn5VOvr7u7D8//8idXQoKacl5uqcLHKT7jZyyvVG4sE9RjooBQQ8vc/4B7sS0I19lRIlfx1WcEJucq04uyngcucFyZGsHJ8ywVk6DFraSMyRWiCOJq+YMx5XzpMxf7XDW1Uv6RmxMM1mcRGCL53xHMaiIrHQ9qpkWlW/5UpdOox1Ux6Tiu1rOkIHN7xRaOHwTg1quAZvGqZSg2Z20HC9ZebW6ZF/yWD+pDe9C7xGy0vqynuIcfpTM6dMAA7ZNtrpJFed1tvUxzuYQt4b4jclqlFyas/iRGJEVQ/lVgbp1eYcq1Uo7o62fWMaBcSSuADlFoHJYt5LCSFfRhDN2EUYYeGk2CZWu/4+M1fba7/4rdP3BfysLXIHVhPP5slHyQofvXmMoDVfRYNno3AZY7U0Hgul85tmfLgCmsmD4F0BL3f7r+NiJY6qpmAg214283wW5QCcLNbcDGvmq11ypfIC7XW0mS2isYTsZjxatPvs++/ZAE/WzDzPykOuurst5vXb1I1RC7fqrvjNIr7koIhVOHydwSghHr7VJhWwdPZhjP/wuR0BR9cwRzqsu9httnkbgDAex/MUJ4OASiZQqPMTrJkOMLOcXoMiS06oCNzvgnJNF7zt7Ypl51t5HpYIIzCZ3WKJBIxiYm7SFB/vnUt4+emPTCfn24jOZ+QnbS/wAuaNPf8r50Yrjy8K6L6Af35C7kEWQtV+ggdHEE3oewMNT7zrmRdQiTTYqXwN6mB98ld4hi8FOOk5GUayBU29RjabZRWhkIkcdCLSCCqNLdyJa7dESy8rAEmaKqicEzVZF3ISDSLvOgXGFQAKjZxjmUD8iasKnVoirkpuiKItvJZQZS2XUHbetNoydylDb2o1qLTj/oaaFq1XVj206noVxclj6K4N9NumNxeEhliUp901BqhOZdFhCU5Va/hCIfynXVJRNuxurvCwPMgLKAAlGHQuPkyv6FK6iNsMN0a8E489Yf1ez9dvivJk+VIlOgTfGbCzf54+v1AvITk9fcfevnvz97PnF2/esV9e/vju9N2/PP9Qz0tP1v0yBn6lfG/FnWeCEF3wBoBUNLKhxJQFVDV9E8pbgbG+BlIAUP3NTN/V3sZkNPaN0pQKkfItAwE4UCpPK3DGfpYppSoNzGX6HW+ACkA7rimCCtBXq3zJ+wBkaAUb6joVeTwBPfHour7XSLdpl/F4MOjLIhp6ZwO9XATPqgVSxaDcfb4TIA/B8rKjF71x9e4luQVcHTZTbznqPuywqls5SytTHZDeovmWqhHrSfv+uHZQqK/5au0B1ZvWYAY6DJaWA4xdxE63NbBhwHoOMLptgg0dIw6tEUdVVyNtxJEONjJGLMFG1Yhy8fnNVtQklJ5z7iRVv06ZQZ0Q/fqk+/UJjurzeVbH/ZnCU+PyCi/wBMb32OJTbh2AB+rdYSqxg5wIhnujikLvUHIc9NAEaeK96IWO0XLPZDD0LWBImIXYmS5RqI4n1scxkbnXoBNvi5RZeMY5OakPX/TY29Pz83FNDiG24TnTFrwfDIJ+MAqeef7Dq4S+QuBXDvYdjVrCjsr3jc1TvlzG8xhfvgVcv8IXXT0sCpJ11JJPBEdP646Ifc5j6E/NYtKqC5IAdxeOKkXo61swIOV1B0sdj30tQp3YzzRoaj0N9OLW68uh2schdEiuaugI1NFNBjYTTT6XvX4DRKAgxhj5uGeNPOr5pgDe6PVQ2KleLEUSpI3YQL2eE936AlgHbcwJ9+7dg6nngPFGYVXo2bBe/YFjwUzSHLjxxYvcr/HIcLT/Dtz4ZuAewYlOrbU5pOiu3uYPDGD21zTJ+9PIid89ZjnV32vl4seRm5ucXOPuZNTIku5ODP3el68e1bShbVK8Xy/DPmbd+0dDPQMHtweheqOCeP5t7XmRRUmu8moI1OkfDY4tsKHRzTdHoB9qEK6Ojh1gq4xzcTBYoDQyYJQjJJ85Ho3CslhSJegakm5O6uo5N2khx2QfdZ9Cmcm+YSZFOKTZKipmrFur/DFs5WDsrpukUpkjei/IfMXnVw9tI6n/MOP5bl3kIjS4p+ekn3iAPvCokbPAp/aOKgl+ck8nzsBwovwrFBH9bRsE5VevV/kz+5lnG2DWOXA1A8YtOkQ/lgKbsAXH91jGCb5GdC6TpeLcSrfsYX/9EWH2paeIZDmRdj4t2kAX6nxaPgEej0FzDUQeXp2kmdrwT8sGHRf8eFovdhItx7VzI7dND/WXijXXVd1S0NkEcEuFV5Jhz9SbchXxI9vxaxiumozlBZjBBFoNv/EW4Wk6CI8z5gB9TNeYHTGoof0H1WuZQyHetu63ZAq3SwyBqCvQEJld9abrPgJaVVKxH/Cg8hwYytPSWHmgy1/gWvPH0KLDMcvn+FZmeod0+XLg6l3S5TbmA6vRsl9tezMfqzfHTsx6M1xi+USln1AUMYlDvjUmc8y9CdpIVc8mXprFYjsVVpse1UoUJFvTw4knXAx8D+CaF+GSg12dYUEBJsVq71ihV4vQNjKqNRxU9iLeC25uMNcGNohRHry1j1dU/ftNSoeOlVV1bnhyqgkUy+ipW5IxkMGnfW2sqpNmRPVVm+TVOtGLbFpNWOmtxCngbwfGObfdxg3cpWQA2FDZ6pvRcNBSJ6h/36nEfLiKqc6k4hWrKnEfmFYBqL+kRlY81rbciVKijtONsrT6mtmnMxI4R1nJuJfkdGyiBO7sB5bb8WDnoVE7WQbQvHpabWYLAHdZq2xkOx9m4zixyh5EiiCQma5AprcCmcAKZNbK8bIWxYoV5prL1DLNya0ewlctKqtd54KJgffUfMOSmyMcTexFb2k0bSojaqyp1ZhJ2Xu7/9rpgdJllzOUnFtPOg579azjX3v1tONxr55kHPTqWcaBbsFKLDR8GzHpfzMcDr6tY9Mf9o8dCH0zcGB07ETp2JHmpJ226jUFFheUQugb0aVGzKa+3CzS0J9FFtNlGYpvPfQrToC1BnML92zfReJVMxnjPWo00GuZCuBd/WsUGiAWVR1Wr0ENmaoBTi2VB+pMJUAxBmZwRRgUwI9KO+5ZFWcVlEXMrxjxwNo5Rz3gvVWAL4bSjVOfVqh9RAEPJPDFd1rZiQcWKwCTJR0r/VMFjHiBy7LKR/DwjscQB0cZ+HXzNPmwuxTHDunkBG4/AWGAULcP7N2R9pvdhrpj89kMA9sIM6nV3039sUBuKT8/Ic6qgrLV1sQyzTqYMWo98lY0qDytzs1+yyqtYNnONU69e9L/ZGot0EkJMq1bWiSIM7YXL3XW+z6UBDD6rCfa7M5cp56UAqMX4gMDhRoDiXIvW3/N12lO6nXBs/Bync5AHeHMw+U63tYjJqvOcZmJ42pg5MOEQz8JRFubqJiv6k0BEvrWYkE92/TlodmxCs0oDNI+foIVPSzSk1kdkflDObbkilOLR4jSfsWimVrYKD+7REk3/XMtDyzKcphwG91iaYy54BuOh13sVwnKWgZ4ICsZrFf+XWbpDtlB1FN4jS8ErJdnVzV/KR2Pw6yR+loAX3P87FGOMRp+HAZswSq9TPFTLfVyzypNwFJ8OVPtNa9j9vbkx/aPv12wH/3fwPuGX2Yv9nsM8cCjPIcIDIEFkoIEjkmcRfMVe49fjHn/RHyZJp7RN20iOlUoXs2Py0qnCWWFoi+zbo65UAn6d+pUOafZ59GGm/uYQBb8hlBarEDPLfAlVHvnI50Wqq9A3wEMrlZr4QYW5Rt0ol7WdAROY6t8+RrrOMqKn4jN4YYarPprcqwkrHQJ7FxszQm4s5i9wBLJqn4GP3yx//tgntGBKGfRihb1ngNbqPxWFYE+pKCZdbpahboFR7FHmM9Tqvp1cOxz4S7sULywbEjzHdXXoeYr0P2Jy9dwcKxIz5j6PxdfgsOqwjXHcq4rXhwBmeBOtt3lpImL63jOXR0iBuT+Ciy/o0rej/ECuImsegctEHhl0reoLA1DS+PqUb7wHiR1xufRLhdiZaLMKkmnr5vltxtYqCyeH5At4fRseJTg4bSx60zU0746VxShVoNpSDX33nX4qUPQqizcbvHE1aRH/UekAqNZTl/0SsSX4TCCsM8qPbJ2sFK3Uj7oNEYlN11UCIfaVMNoN9xqSPgwwt/MdTWgBNGhBPZ97s/TGtsKAGvVqm4DU9ZLl+EdFsl18PhSVxZxUrGbSk0ZlW9Vb7WX7OnNJnKa06aE23xy74Mn4iDLnEoY9gxB2TltlLKuNjfUINWIAyFvCtDHE6HiaFcYvQPyUqWDYLqjz9THj7J0J896Ne8BaJaGTm/ISyNq1lZSgtFv//DJnAeIsl2utdKA7v7xbcJfNsZB9/eZdH//fv7m9RHWUbOsZETtDTUZn6elHfDKQ4vRNreLb/FFkaIoV742Tnzm8iFsGp4ZQNVVfdHNdiDFN1dxJW6BHqI7G0R+2A5PicqfXfWjbauva4j0cVohUl9O1wJR9bSI2PO3tor2on6vF6LNiTNaByrDd+skpVEtqqC8HNazhxSsKIYdaxWwDYpRrHANC7EtUpcusAPz9Q7L7EtG3idy1gEvp7Q+s8BlFsm9peKW2n2S+0BJssMi3DzQF8uxEJroRjqhrveMlSDuQ2CjJjto7QeUeTcHD9hpwdzpv/zhjKDL6TDp94WD3jcpaA2s08lReDQ+UJekH4iTQkH/68FQvHbMp+b3OL3E/b6OK97aI5IWuSv/4h7jl06P82WGzZZ1D7VVOPLZjlWkaDnfvODZBbzVt4Vh+kET/AvgdeEuY0SDFk/mwpxtrNfVawVQ+uKDLw1W9HKl6mKGT4euzrS336vSqSqyJEZW7QmvkasPTEXv/7CvyEnj9331VJHVl3UsG0Og8AC5RZKPub+oqz66Gze8sYFIv0x3WbHqUJDKLnmKAdQtw/qauNgt3Avg/NAuUiffzdSI4sO7rtY/H6Mjve7kWyyrmu9mvDNDdyei1F9eNFPlzjqfmoFNKT9vYfHp5XYHKnoe541pID3a1XcGVGJPpucxbbO+xYRXRkO581kl89KLxz9xFkH4muYFG92MuuJAD8awGccjVjnbga3KIVqM+eJArIrfURM7WA2pAe3LytJPFHPRPktP4YsSB/qUdMkvrlje8VFmlAMwUsDKKZXEkPcSyBdLoG3HVCB+IbghNzDHD/HibOn01sBnC/yUVcrkp17REDPK9CbgAWKeSaQuUFc30+fg/pHu+zaGk44vwXtGw1ouSe8xUGDG8TXr1NqrV/YpNO0kqWw/UV7ftPmYrO83jLH05FE3PHWmmy77dNrz03d/OxeH0yr7YgNJhBFIn2oT2PnPp2CENGAgmA376+kvv1y8fHUGUNKDH3cHyzuWm+fiAKo81/fi9OLsnAIjr2na1fK/PvvnharskjqxQRNilgVChIDN+GWcYAJIHF/VdguRYU02zdAE5HRAEFieY0TSLXcMW/GSheKVRCFFzWGIQh6G8p0x9CnwDD/sJz8L3j3NLneYI3pLT9paBaf4gDyMc+KdabIsDJ1Vaav0FMrmv44VOtWI3WixCCM5VNvrdERsE+DHvPmJONkpv2d8Yn21em83QoI6IEH7+9K/kSxTJNklKI1wl1wl6XUi3vWPI9B/Id3EkdTBIjqNitddQiug9t2qW7/1f0amXVjugwAA'


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def decode_source(payload: str) -> bytes:
    return gzip.decompress(base64.b64decode(payload.encode("ascii")))


def default_stage0() -> Path:
    candidates = [
        Path("/content/Y4_STAGE0/y4_ordered_transition_words.json.gz"),
        Path.cwd() / "Y4_STAGE0" / "y4_ordered_transition_words.json.gz",
        Path("/mnt/data/Y4_STAGE0_TEST/y4_ordered_transition_words.json.gz"),
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


def default_root() -> Path:
    return Path("/content") if Path("/content").exists() else Path.cwd()


def run_checked(cmd):
    print("\n[autobundle] RUN:", " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(x) for x in cmd], check=True)


def validate_stage1(path: Path) -> None:
    if not path.exists():
        raise RuntimeError(f"Stage 1 did not create expected file: {path}")
    with gzip.open(path, "rt", encoding="utf-8") as f:
        obj = json.load(f)
    words = obj.get("words", [])
    if len(words) != 4221:
        raise RuntimeError(f"Stage-1 manifest has {len(words)} words, expected 4221")


def validate_stage2(outdir: Path) -> None:
    summary = outdir / "y4_stage2_summary.json"
    cards = outdir / "y4_link_tensor_cards.json.gz"
    library = outdir / "su3_local_haar_projectors.json"
    for p in (summary, cards, library):
        if not p.exists():
            raise RuntimeError(f"Stage 2 did not create expected file: {p}")
    data = json.loads(summary.read_text(encoding="utf-8"))
    if data.get("passed") is not True:
        raise RuntimeError("Stage-2 summary does not report passed=true")
    counts = data.get("counts", {})
    if counts.get("unique_link_token_signatures") != 182:
        raise RuntimeError(
            f"Unexpected Stage-2 signature count: {counts.get('unique_link_token_signatures')}"
        )
    representative_count = counts.get("c_orbit_representative_link_tensor_occurrences")
    if representative_count != 187632:
        raise RuntimeError(
            f"Unexpected Stage-2 tensor count: {representative_count}"
        )


def main(stage0: Path, root: Path, force_stage1: bool, force_stage2: bool) -> None:
    print("=" * 100)
    print("Y4 STAGE-1 + STAGE-2 SELF-CONTAINED COLAB AUTOBUNDLE")
    print("=" * 100)
    print("version :", VERSION)
    print("stage0  :", stage0)
    print("root    :", root)
    print("hardware: CPU; A100 not used")

    if not stage0.exists():
        checked = [
            "/content/Y4_STAGE0/y4_ordered_transition_words.json.gz",
            str(Path.cwd() / "Y4_STAGE0" / "y4_ordered_transition_words.json.gz"),
        ]
        raise FileNotFoundError(
            "The completed Stage-0 ordered-word manifest was not found. Expected one of:\n  - "
            + "\n  - ".join(checked)
        )

    stage1_dir = root / "Y4_STAGE1"
    stage2_dir = root / "Y4_STAGE2"
    stage1_dir.mkdir(parents=True, exist_ok=True)
    stage2_dir.mkdir(parents=True, exist_ok=True)
    stage1_manifest = stage1_dir / "y4_channel_denominator_manifest.json.gz"

    with tempfile.TemporaryDirectory(prefix="y4_bundle_") as td:
        td = Path(td)
        stage1_script = td / "y4_stage1_channel_denominator_manifest.py"
        stage2_script = td / "y4_stage2_exact_haar_library.py"
        stage1_script.write_bytes(decode_source(STAGE1_GZ_B64))
        stage2_script.write_bytes(decode_source(STAGE2_GZ_B64))

        if force_stage1 or not stage1_manifest.exists():
            print("\n[autobundle] Stage-1 manifest is absent; rebuilding it from Stage 0.", flush=True)
            run_checked([
                sys.executable, "-u", stage1_script,
                "--input", stage0,
                "--output-dir", stage1_dir,
            ])
        else:
            print("\n[autobundle] Reusing existing Stage-1 manifest:", stage1_manifest, flush=True)

        validate_stage1(stage1_manifest)
        print("[autobundle] Stage-1 validation PASS")
        print("[autobundle] Stage-1 SHA256:", sha256(stage1_manifest))

        stage2_summary = stage2_dir / "y4_stage2_summary.json"
        if force_stage2 or not stage2_summary.exists():
            run_checked([
                sys.executable, "-u", stage2_script,
                "--input", stage1_manifest,
                "--output-dir", stage2_dir,
            ])
        else:
            print("\n[autobundle] Reusing existing Stage-2 output:", stage2_summary, flush=True)

    validate_stage2(stage2_dir)
    print("\n[autobundle] Stage-2 validation PASS")
    print("[autobundle] Stage-2 summary SHA256:", sha256(stage2_dir / "y4_stage2_summary.json"))
    print("\nALL AUTOBUNDLE GATES PASS")
    print("STAGE1:", stage1_dir)
    print("STAGE2:", stage2_dir)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage0", type=Path, default=default_stage0())
    ap.add_argument("--root", type=Path, default=default_root())
    ap.add_argument("--force-stage1", action="store_true")
    ap.add_argument("--force-stage2", action="store_true")
    args, _ = ap.parse_known_args()
    main(args.stage0, args.root, args.force_stage1, args.force_stage2)



Y4 STAGE-1 + STAGE-2 SELF-CONTAINED COLAB AUTOBUNDLE
version : 2026-06-13-autobundle-v1
stage0  : /content/Y4_STAGE0/y4_ordered_transition_words.json.gz
root    : /content
hardware: CPU; A100 not used

[autobundle] Stage-1 manifest is absent; rebuilding it from Stage 0.

[autobundle] RUN: /usr/bin/python3 -u /tmp/y4_bundle_q93acpjz/y4_stage1_channel_denominator_manifest.py --input /content/Y4_STAGE0/y4_ordered_transition_words.json.gz --output-dir /content/Y4_STAGE1
[autobundle] Stage-1 validation PASS
[autobundle] Stage-1 SHA256: 4e579d1dd74d208b9c4ab6bc596408936f4e244e9cd588a8f5438d6e20940565

[autobundle] RUN: /usr/bin/python3 -u /tmp/y4_bundle_q93acpjz/y4_stage2_exact_haar_library.py --input /content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz --output-dir /content/Y4_STAGE2

[autobundle] Stage-2 validation PASS
[autobundle] Stage-2 summary SHA256: 3e659ef7f32bbbc2a7a954807e788eb8b100c484dc0429ebe357f15d93beb81e

ALL AUTOBUNDLE GATES PASS
STAGE1: /content/Y4_STAGE1
STAGE2: /c

In [9]:
#!/usr/bin/env python3
"""
y4_stage3a_domino_recoupling_firewall.py
========================================

Exact SU(3) time-resolved recoupling firewall for the O(y^4) flat-band program.

RUN
---
    %run /content/y4_stage3a_domino_recoupling_firewall.py

HARDWARE
--------
Use a standard Colab CPU runtime. The A100 is not used.

PURPOSE
-------
Stage 2 constructed FINAL local Haar projectors. Those projectors are not enough
to combine fourth-order amplitudes with resolvent denominators, because the
denominators depend on INTERMEDIATE SU(3) representation channels.

This Stage 3A freezes the required time-resolved channel convention by
constructing the exact shared-link projectors

    3 ⊗ 3bar = 1 ⊕ 8
    3 ⊗ 3    = 3bar ⊕ 6

and reproducing the complete certified O(y^2) domino constants:

    channel sum        = -481/612
    mixed family       = -27/68
    like family        = -7/18
    C-even hop         = -11/306
    C-odd signed hop   =  5 s / 612
    even domino levels = {1769/3060, 13/20}
    odd domino levels  = {31/68, 17/36}

Only after this firewall passes may a fourth-order time-ordered recoupling
engine be trusted.

OPTIONAL INPUT VALIDATION
-------------------------
If the Stage-1 and Stage-2 outputs exist, this script validates their headline
counts and projector coefficients. It remains self-contained if they do not.

OUTPUT
------
    /content/Y4_STAGE3A/y4_stage3a_domino_recoupling.json
    /content/Y4_STAGE3A/y4_stage3a_summary.json
"""

from __future__ import annotations

import argparse
import gzip
import hashlib
import json
import platform
import sys
import time
from fractions import Fraction
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

VERSION = "2026-06-13-stage3a-v1"
N = 3

Matrix = List[List[Fraction]]
Pair = Tuple[int, int]


def default_output_dir() -> Path:
    if Path("/content").exists():
        return Path("/content/Y4_STAGE3A")
    return Path.cwd() / "Y4_STAGE3A"


def find_optional(paths: Sequence[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None


def frac(x) -> Fraction:
    return x if isinstance(x, Fraction) else Fraction(x)


def fstr(x: Fraction) -> str:
    return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"


def zero_matrix(n: int) -> Matrix:
    return [[Fraction(0) for _ in range(n)] for _ in range(n)]


def identity_matrix(n: int) -> Matrix:
    return [[Fraction(int(i == j)) for j in range(n)] for i in range(n)]


def madd(a: Matrix, b: Matrix) -> Matrix:
    return [[a[i][j] + b[i][j] for j in range(len(a[0]))] for i in range(len(a))]


def msub(a: Matrix, b: Matrix) -> Matrix:
    return [[a[i][j] - b[i][j] for j in range(len(a[0]))] for i in range(len(a))]


def mmul(a: Matrix, b: Matrix) -> Matrix:
    return [
        [
            sum(a[i][k] * b[k][j] for k in range(len(b)))
            for j in range(len(b[0]))
        ]
        for i in range(len(a))
    ]


def mscale(c: Fraction, a: Matrix) -> Matrix:
    return [[c * x for x in row] for row in a]


def mtrace(a: Matrix) -> Fraction:
    return sum(a[i][i] for i in range(len(a)))


def mzero(a: Matrix) -> bool:
    return all(x == 0 for row in a for x in row)


def matrix_strings(a: Matrix) -> List[List[str]]:
    return [[fstr(x) for x in row] for row in a]


PAIRS: List[Pair] = [(i, j) for i in range(N) for j in range(N)]
PAIR_INDEX: Dict[Pair, int] = {p: k for k, p in enumerate(PAIRS)}


def delta(i: int, j: int) -> int:
    return int(i == j)


# ---------------------------------------------------------------------------
# Exact shared-link channel projectors
# ---------------------------------------------------------------------------

def projector_3x3bar_singlet() -> Matrix:
    """
    Basis |i, jbar>.  Singlet vector is (1/sqrt(3)) sum_i |i, ibar>.
    P1[(i,j),(k,l)] = delta_ij delta_kl / 3.
    """
    p = zero_matrix(9)
    for i, j in PAIRS:
        for k, l in PAIRS:
            p[PAIR_INDEX[(i, j)]][PAIR_INDEX[(k, l)]] = Fraction(
                delta(i, j) * delta(k, l), 3
            )
    return p


def swap_matrix() -> Matrix:
    s = zero_matrix(9)
    for i, j in PAIRS:
        s[PAIR_INDEX[(i, j)]][PAIR_INDEX[(j, i)]] = Fraction(1)
    return s


def channel_projectors() -> Dict[str, Matrix]:
    identity = identity_matrix(9)

    p1 = projector_3x3bar_singlet()
    p8 = msub(identity, p1)

    swap = swap_matrix()
    pbar3 = mscale(Fraction(1, 2), msub(identity, swap))
    p6 = mscale(Fraction(1, 2), madd(identity, swap))

    return {
        "1": p1,
        "8": p8,
        "bar3": pbar3,
        "6": p6,
    }


# ---------------------------------------------------------------------------
# SU(3) representation data
# ---------------------------------------------------------------------------

def dim_irrep(p: int, q: int) -> int:
    return (p + 1) * (q + 1) * (p + q + 2) // 2


def c2(p: int, q: int) -> Fraction:
    return Fraction(p * p + q * q + p * q + 3 * p + 3 * q, 3)


IRREPS = {
    "1": (0, 0),
    "8": (1, 1),
    "bar3": (0, 1),
    "6": (2, 0),
}

EXPECTED_DIMS = {"1": 1, "8": 8, "bar3": 3, "6": 6}

# The two-plaquette intermediate has six free fundamental links:
# 6 * (1/2) C2(3) = 6 * 2/3 = 4.
FREE_LINK_ENERGY = Fraction(4, 1)
E0 = Fraction(8, 3)


def channel_energy(name: str) -> Fraction:
    p, q = IRREPS[name]
    return FREE_LINK_ENERGY + Fraction(1, 2) * c2(p, q)


# ---------------------------------------------------------------------------
# I/O
# ---------------------------------------------------------------------------

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, obj: object) -> str:
    raw = json.dumps(obj, indent=2, sort_keys=True, allow_nan=False).encode("utf-8")
    path.write_bytes(raw)
    return hashlib.sha256(raw).hexdigest()


def read_json_gz(path: Path) -> object:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main(output_dir: Path) -> None:
    started = time.time()
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("SU(3) O(y^4) STAGE-3A TIME-RESOLVED DOMINO RECOUPLING FIREWALL")
    print("=" * 100)
    print(f"version  : {VERSION}")
    print(f"output   : {output_dir}")
    print("hardware : standard Colab CPU; A100 is not used")
    print()

    gates = {}

    # ------------------------------------------------------------------
    # G0: optional Stage-1 / Stage-2 dependency validation.
    # ------------------------------------------------------------------
    stage1_path = find_optional([
        Path("/content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz"),
        Path.cwd() / "Y4_STAGE1" / "y4_channel_denominator_manifest.json.gz",
    ])
    stage2_path = find_optional([
        Path("/content/Y4_STAGE2/su3_local_haar_projectors.json"),
        Path.cwd() / "Y4_STAGE2" / "su3_local_haar_projectors.json",
    ])

    dependency = {
        "stage1_present": stage1_path is not None,
        "stage2_present": stage2_path is not None,
    }

    if stage1_path is not None:
        stage1 = read_json_gz(stage1_path)
        assert len(stage1["words"]) == 4221
        dependency["stage1_path"] = str(stage1_path)
        dependency["stage1_sha256"] = sha256_file(stage1_path)
        dependency["stage1_words"] = 4221

    if stage2_path is not None:
        stage2 = json.loads(stage2_path.read_text(encoding="utf-8"))
        fam = stage2["families"]
        assert fam["(1,1)"]["gram_inverse"] == [["1/3"]]
        assert fam["(2,2)"]["gram_inverse"] == [
            ["1/8", "-1/24"],
            ["-1/24", "1/8"],
        ]
        dependency["stage2_path"] = str(stage2_path)
        dependency["stage2_sha256"] = sha256_file(stage2_path)
        dependency["stage2_projector_families"] = len(fam)

    gates["G0_dependencies"] = {**dependency, "passed": True}
    print(
        "G0 PASS: optional Stage-1/Stage-2 validation "
        f"(stage1={dependency['stage1_present']}, stage2={dependency['stage2_present']})"
    )

    # ------------------------------------------------------------------
    # G1: exact channel projectors.
    # ------------------------------------------------------------------
    projectors = channel_projectors()
    identity = identity_matrix(9)

    assert mzero(msub(madd(projectors["1"], projectors["8"]), identity))
    assert mzero(msub(madd(projectors["bar3"], projectors["6"]), identity))

    for name, p in projectors.items():
        assert mzero(msub(mmul(p, p), p)), f"{name} not idempotent"
        assert mtrace(p) == EXPECTED_DIMS[name]

    assert mzero(mmul(projectors["1"], projectors["8"]))
    assert mzero(mmul(projectors["8"], projectors["1"]))
    assert mzero(mmul(projectors["bar3"], projectors["6"]))
    assert mzero(mmul(projectors["6"], projectors["bar3"]))

    gates["G1_channel_projectors"] = {
        name: {
            "trace": fstr(mtrace(p)),
            "expected_dimension": EXPECTED_DIMS[name],
            "idempotent": True,
        }
        for name, p in projectors.items()
    }
    gates["G1_channel_projectors"]["passed"] = True
    print("G1 PASS: exact orthogonal projectors for 1, 8, bar3, 6")

    # ------------------------------------------------------------------
    # G2: representation dimensions and Casimirs.
    # ------------------------------------------------------------------
    irrep_table = {}
    for name, (p, q) in IRREPS.items():
        d = dim_irrep(p, q)
        c = c2(p, q)
        assert d == EXPECTED_DIMS[name]
        irrep_table[name] = {
            "dynkin": [p, q],
            "dimension": d,
            "C2": fstr(c),
            "half_C2": fstr(Fraction(1, 2) * c),
        }

    assert c2(1, 0) == Fraction(4, 3)
    assert c2(1, 1) == Fraction(3, 1)
    assert c2(2, 0) == Fraction(10, 3)

    gates["G2_irrep_data"] = {**irrep_table, "passed": True}
    print("G2 PASS: exact SU(3) dimensions and quadratic Casimirs")

    # ------------------------------------------------------------------
    # G3: exact channel norms and energies.
    # ------------------------------------------------------------------
    channels = {}
    expected_energies = {
        "1": Fraction(4, 1),
        "8": Fraction(11, 2),
        "bar3": Fraction(14, 3),
        "6": Fraction(17, 3),
    }

    for name in ("1", "8", "bar3", "6"):
        norm = mtrace(projectors[name]) / 9
        energy = channel_energy(name)
        denominator = E0 - energy
        term = norm / denominator

        assert norm == Fraction(EXPECTED_DIMS[name], 9)
        assert energy == expected_energies[name]

        channels[name] = {
            "dimension": EXPECTED_DIMS[name],
            "norm": fstr(norm),
            "energy": fstr(energy),
            "denominator_E0_minus_E": fstr(denominator),
            "second_order_term": fstr(term),
        }

    expected_terms = {
        "1": Fraction(-1, 12),
        "8": Fraction(-16, 51),
        "bar3": Fraction(-1, 6),
        "6": Fraction(-2, 9),
    }
    assert {
        name: Fraction(channels[name]["second_order_term"])
        for name in channels
    } == expected_terms

    gates["G3_channel_terms"] = {**channels, "passed": True}
    print("G3 PASS: exact channel norms, energies, and resolvent terms")

    # ------------------------------------------------------------------
    # G4: certified domino constants.
    # ------------------------------------------------------------------
    mixed = expected_terms["1"] + expected_terms["8"]
    like = expected_terms["bar3"] + expected_terms["6"]
    channel_sum = mixed + like

    assert mixed == Fraction(-27, 68)
    assert like == Fraction(-7, 18)
    assert channel_sum == Fraction(-481, 612)

    # C-even vacuum-intermediate route:
    # |<0|W|e>|^2 = 2 and E0 - Evac = 8/3.
    vacuum_matrix_element_squared = Fraction(2, 1)
    vacuum_route = vacuum_matrix_element_squared / E0
    even_hop = channel_sum + vacuum_route

    # C-odd orientation-sensitive hopping is the family difference.
    odd_hop_coefficient = like - mixed

    assert vacuum_route == Fraction(3, 4)
    assert even_hop == Fraction(-11, 306)
    assert odd_hop_coefficient == Fraction(5, 612)

    gates["G4_domino_constants"] = {
        "mixed_1_plus_8": fstr(mixed),
        "like_bar3_plus_6": fstr(like),
        "channel_sum": fstr(channel_sum),
        "vacuum_matrix_element_squared": fstr(vacuum_matrix_element_squared),
        "vacuum_route": fstr(vacuum_route),
        "C_even_hop": fstr(even_hop),
        "C_odd_hop": "s * 5/612",
        "passed": True,
    }
    print(
        "G4 PASS: channel sum=-481/612, "
        "t_even=-11/306, t_odd=s*5/612"
    )

    # ------------------------------------------------------------------
    # G5: independent two-plaquette spectral anchors.
    # ------------------------------------------------------------------
    even_diag = Fraction(1879, 3060)
    even_levels = sorted([even_diag - abs(even_hop), even_diag + abs(even_hop)])
    assert even_levels == sorted([Fraction(1769, 3060), Fraction(13, 20)])

    odd_diag = Fraction(71, 153)
    odd_levels = sorted([
        odd_diag - odd_hop_coefficient,
        odd_diag + odd_hop_coefficient,
    ])
    assert odd_levels == sorted([Fraction(31, 68), Fraction(17, 36)])

    gates["G5_domino_spectra"] = {
        "even_diag": fstr(even_diag),
        "even_half_splitting": fstr(abs(even_hop)),
        "even_levels": [fstr(x) for x in even_levels],
        "odd_diag": fstr(odd_diag),
        "odd_half_splitting": fstr(odd_hop_coefficient),
        "odd_levels": [fstr(x) for x in odd_levels],
        "passed": True,
    }
    print("G5 PASS: exact C-even and C-odd domino spectra reproduced")

    # ------------------------------------------------------------------
    # G6: normalization identities relevant to Stage 3B.
    # ------------------------------------------------------------------
    assert sum(Fraction(channels[n]["norm"]) for n in channels) == Fraction(2, 1)
    assert mixed - like == Fraction(-5, 612)
    assert like - mixed == Fraction(5, 612)

    gates["G6_normalization"] = {
        "sum_all_channel_norms": "2",
        "mixed_minus_like": "-5/612",
        "like_minus_mixed": "5/612",
        "resolvent_convention": "1/(E0-E_intermediate)",
        "bra_conjugation_required": True,
        "passed": True,
    }
    print("G6 PASS: normalization and sign conventions frozen")

    payload = {
        "meta": {
            "version": VERSION,
            "date": "2026-06-13",
            "python": sys.version,
            "platform": platform.platform(),
            "hardware": "CPU",
            "a100_required": False,
        },
        "dependencies": dependency,
        "representation_data": irrep_table,
        "projectors": {
            name: matrix_strings(p) for name, p in projectors.items()
        },
        "channels": channels,
        "certified_constants": gates["G4_domino_constants"],
        "spectral_anchors": gates["G5_domino_spectra"],
        "gates": gates,
        "scope": {
            "completed": [
                "exact shared-link SU(3) channel projectors",
                "time-resolved channel normalization",
                "intermediate Casimir energies",
                "second-order resolvent convention",
                "C-even vacuum route",
                "C-odd orientation-family difference",
                "domino spectrum regression firewall",
            ],
            "not_completed": [
                "fourth-order recoupling tensors",
                "time-ordered intertwiner matrices for all Stage-1 words",
                "des-Cloizeaux folded terms",
                "fourth-order flat-band residual",
            ],
        },
        "next_stage": (
            "Build Stage 3B time-ordered link transfer matrices carrying both "
            "intermediate irrep labels and exact intertwiner amplitudes. "
            "The final Stage-2 Haar projector must not be multiplied directly "
            "by Stage-1 denominator multiplicities."
        ),
        "passed": True,
    }

    detail_path = output_dir / "y4_stage3a_domino_recoupling.json"
    detail_sha = write_json(detail_path, payload)

    elapsed = time.time() - started
    summary = {
        "meta": {
            "version": VERSION,
            "walltime_s": elapsed,
            "hardware": "CPU",
            "a100_required": False,
        },
        "headline": {
            "channel_sum": "-481/612",
            "C_even_hop": "-11/306",
            "C_odd_hop": "s * 5/612",
            "even_levels": ["1769/3060", "13/20"],
            "odd_levels": ["31/68", "17/36"],
        },
        "detail_file": {
            "path": str(detail_path),
            "sha256": detail_sha,
        },
        "all_gates_passed": True,
    }

    summary_path = output_dir / "y4_stage3a_summary.json"
    summary_sha = write_json(summary_path, summary)

    print()
    print("SUMMARY")
    print(json.dumps(summary["headline"], indent=2, sort_keys=True))
    print()
    print(f"DETAIL : {detail_path}")
    print(f"SUMMARY: {summary_path}")
    print(f"SUMMARY SHA256: {summary_sha}")
    print(f"WALLTIME: {elapsed:.3f} s")
    print("ALL STAGE-3A GATES PASS")
    print()
    print(
        "NEXT: Stage 3B exact time-ordered intertwiner transfer matrices. "
        "Continue on CPU."
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Exact SU(3) domino recoupling firewall"
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=default_output_dir(),
    )
    args, _unknown = parser.parse_known_args()
    main(args.output_dir)


SU(3) O(y^4) STAGE-3A TIME-RESOLVED DOMINO RECOUPLING FIREWALL
version  : 2026-06-13-stage3a-v1
output   : /content/Y4_STAGE3A
hardware : standard Colab CPU; A100 is not used

G0 PASS: optional Stage-1/Stage-2 validation (stage1=True, stage2=True)
G1 PASS: exact orthogonal projectors for 1, 8, bar3, 6
G2 PASS: exact SU(3) dimensions and quadratic Casimirs
G3 PASS: exact channel norms, energies, and resolvent terms
G4 PASS: channel sum=-481/612, t_even=-11/306, t_odd=s*5/612
G5 PASS: exact C-even and C-odd domino spectra reproduced
G6 PASS: normalization and sign conventions frozen

SUMMARY
{
  "C_even_hop": "-11/306",
  "C_odd_hop": "s * 5/612",
  "channel_sum": "-481/612",
  "even_levels": [
    "1769/3060",
    "13/20"
  ],
  "odd_levels": [
    "31/68",
    "17/36"
  ]
}

DETAIL : /content/Y4_STAGE3A/y4_stage3a_domino_recoupling.json
SUMMARY: /content/Y4_STAGE3A/y4_stage3a_summary.json
SUMMARY SHA256: 0e4f9540d69591825dbae78b71a8dea56f53213c9a5d91a3e36f70d0fd50c8ea
WALLTIME: 2.766 s